<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_3/bonus/duckdb_data_lake/note_bonus_duckdb_data_lake_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦆 Бонус — DuckDB і data lake: патерни зберігання та аналізу даних

Диспетчерська «Смачно + Таксі» вже має **PostgreSQL** (урок 30) для замовлень, що відбуваються просто зараз, і **Redis** (урок 31) для кешу й черг. Але власниця тепер питає інше:

- «Який виторг по кожному району за два місяці? А як він змінювався день у день?»
- «Хто три найкращі кур'єри в кожному районі?»
- «Збережіть усю історію замовлень і поїздок таксі, навіть «брудну», — аналітики потім розберуться».

Це **аналітика**: кілька великих запитів, кожен проходить сотні тисяч рядків. Для неї є окремий інструмент — **DuckDB**, і окремий спосіб зберігати дані — **data lake** (озеро даних): файли Parquet у папках, а не таблиці на сервері.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | підготовка: пакети, синтетичні дані «Смачно + Таксі» | 0.1–0.2 | |
| 1 | **OLTP і OLAP**: рядки проти стовпців, вимірюємо SQLite і DuckDB | 1.1–1.4 | |
| 2 | **основи DuckDB**: `duckdb.sql`, запит до DataFrame, `.df()` / `.fetchall()`, файл бази, параметри `?`, `GROUP BY ALL`, `SUMMARIZE`, `PIVOT`, `EXPLAIN` | 2.1–2.10 | |
| 3 | **віконні функції**: накопичений підсумок, `rank`, `QUALIFY`, `lag` | 3.1–3.4 | **Задача 1** |
| 4 | **pandas і DuckDB**: той самий звіт двома способами, Arrow | 4.1–4.2 | |
| 5 | **формати файлів**: CSV проти Parquet, projection і predicate pushdown, JSON | 5.1–5.7 | |
| 6 | **data lake**: schema-on-write і schema-on-read, Hive-розбиття, проблема дрібних файлів | 6.1–6.4 | |
| 7 | **медальйонна архітектура**: bronze → silver, пакет конвеєра, тести, ідемпотентне й інкрементне завантаження, перевірки якості | 7.1–7.10 | **Задача 2** |
| 8 | **gold і зіркова схема**: виміри й факти, вітрини, каталог даних | 8.1–8.5 | **Задача 3** |
| 9 | **архітектура**: що де живе, коли DuckDB не підходить | 9.1–9.2 | |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, константи, SQL-рядки) лише створює їх — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Час виконання в тебе буде іншим, ніж у тексті, — важливе **співвідношення**: «у 30 разів швидше», а не «0,004 с».
- Усі файли ноутбук створює сам у папці `duckdb_lab/` поруч із собою; мережа не потрібна. Перший запуск стирає цю папку — так кожен запуск починається з чистого озера.
- Теорія, схеми й таблиці порівнянь — у книзі: [Бонус. DuckDB і data lake](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m3/bonus_duckdb/).

---
## 0. Підготовка

### Приклад 0.1. Пакети

Потрібні `duckdb`, `pyarrow`, `pandas` і `pytest`. У Google Colab вони вже стоять, але DuckDB там може бути старішим — клітинка перевіряє версії **до** імпорту і встановлює лише те, чого бракує. Синтаксис ноутбука перевірено на DuckDB 1.5.

In [ ]:
import importlib.metadata
import subprocess
import sys

NEED = {"duckdb": (1, 5), "pyarrow": (14, 0), "pandas": (2, 0), "pytest": (7, 0)}


def installed_version(package):
    """Версія пакета як кортеж (1, 5) або None, якщо пакета немає. Пакет при цьому не імпортується."""
    try:
        return tuple(int(part) for part in importlib.metadata.version(package).split(".")[:2])
    except (importlib.metadata.PackageNotFoundError, ValueError):
        return None

In [ ]:
missing = [f"{name}>={'.'.join(map(str, version))}" for name, version in NEED.items()
           if (installed_version(name) or (0, 0)) < version]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
print("встановлено:", ", ".join(f"{name} {importlib.metadata.version(name)}" for name in NEED))

In [ ]:
import json
import os
import shutil
import sqlite3
import time
from datetime import date
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa

LAB = Path("duckdb_lab")             # усі файли уроку — тут
if LAB.exists():
    shutil.rmtree(LAB)               # кожен запуск — з чистого озера
LAB.mkdir()
print("DuckDB", duckdb.__version__, "| робоча папка:", LAB)

### Приклад 0.2. Синтетичні дані «Смачно + Таксі»

Справжніх замовлень у нас немає, тому **згенеруємо** їх: 36 ресторанів (6 кухонь × 6 районів Києва), 60 кур'єрів і водіїв, замовлення їжі й поїздки таксі за вересень–жовтень 2026 року. Генератор отримує `rng = np.random.default_rng(42)`: однакове зерно → однакові дані при кожному запуску, тож числа в тебе збігатимуться з книгою.

Кілька правил, щоб дані були схожі на справжні: у вихідні замовлень на 30 % більше; ресторани мають різну популярність; середній чек залежить від кухні (суші дорожчі за борщ); 90 % замовлень доставлені, решта скасовані чи повернені.

In [ ]:
DISTRICTS = ["Поділ", "Оболонь", "Печерськ", "Дарниця", "Святошин", "Голосіїв"]
CUISINES = {"Піца": 320, "Суші": 540, "Борщ": 210, "Бургер": 290, "Вега": 260, "Хінкалі": 380}   # типовий чек, грн
NAMES = ["Оксана", "Тарас", "Ігор", "Марія", "Андрій", "Олена", "Богдан", "Ірина", "Максим", "Софія",
         "Дмитро", "Наталія", "Юрій", "Катерина", "Олег", "Анна", "Віктор", "Леся", "Роман", "Галина",
         "Петро", "Христина", "Василь", "Ольга", "Степан", "Дарина", "Микола", "Яна", "Сергій", "Вікторія"]
START = date(2026, 9, 1)
DAYS = 61                                       # вересень і жовтень 2026


def make_restaurants():
    pairs = [(district, cuisine) for district in DISTRICTS for cuisine in CUISINES]
    return pd.DataFrame([(i, f"{cuisine} {district}", cuisine, district) for i, (district, cuisine) in enumerate(pairs, start=1)],
                        columns=["restaurant_id", "name", "cuisine", "district"])


def make_drivers(rng):
    return pd.DataFrame({
        "driver_id": np.arange(1, 61),
        "name": [f"{NAMES[i % 30]} {'КМ'[i // 30]}." for i in range(60)],
        "district": [DISTRICTS[i % 6] for i in range(60)],
        "vehicle": rng.choice(["авто", "скутер", "велосипед"], size=60, p=[0.5, 0.3, 0.2]),
    })


def make_orders(restaurants, drivers, rng, per_day=5000):
    days = pd.date_range(START, periods=DAYS, freq="D")
    counts = rng.poisson(per_day * np.where(days.dayofweek >= 5, 1.3, 1.0))     # вихідні +30 %
    n = int(counts.sum())
    ts = (np.repeat(days.values, counts) + rng.integers(9 * 3600, 23 * 3600, n).astype("timedelta64[s]")).astype("datetime64[us]")
    popularity = rng.dirichlet(np.full(len(restaurants), 2.0))                  # ресторани різні за популярністю
    rest = rng.choice(len(restaurants), size=n, p=popularity)
    base = restaurants["cuisine"].map(CUISINES).to_numpy()
    df = pd.DataFrame({
        "ts": ts,
        "restaurant_id": restaurants["restaurant_id"].to_numpy()[rest],
        "driver_id": rng.integers(1, len(drivers) + 1, n),
        "customer_id": rng.integers(1, 40_001, n),
        "items": rng.integers(1, 7, n),
        "total": np.round(base[rest] * rng.lognormal(0, 0.35, n), 2),
        "status": rng.choice(["delivered", "cancelled", "returned"], size=n, p=[0.9, 0.07, 0.03]),
    }).sort_values("ts", ignore_index=True)
    df.insert(0, "order_id", np.arange(1, n + 1))
    df.insert(2, "order_date", df["ts"].dt.date)
    return df


def make_rides(drivers, rng, per_day=2500):
    days = pd.date_range(START, periods=DAYS, freq="D")
    counts = rng.poisson(per_day * np.where(days.dayofweek >= 4, 1.2, 1.0))     # п'ятниця–неділя +20 %
    n = int(counts.sum())
    ts = (np.repeat(days.values, counts) + rng.integers(0, 24 * 3600, n).astype("timedelta64[s]")).astype("datetime64[us]")
    taxi = drivers.loc[drivers["vehicle"] == "авто", "driver_id"].to_numpy()
    distance = np.round(rng.gamma(2.0, 3.0, n) + 0.5, 1)
    df = pd.DataFrame({
        "ts": ts,
        "driver_id": rng.choice(taxi, size=n),
        "from_district": rng.choice(DISTRICTS, size=n),
        "to_district": rng.choice(DISTRICTS, size=n),
        "distance_km": distance,
        "fare": np.round(60 + distance * 18 * rng.uniform(0.9, 1.4, n), 2),
    }).sort_values("ts", ignore_index=True)
    df.insert(0, "ride_id", np.arange(1, n + 1))
    df.insert(2, "ride_date", df["ts"].dt.date)
    return df

In [ ]:
rng = np.random.default_rng(42)
restaurants = make_restaurants()
drivers = make_drivers(rng)
orders = make_orders(restaurants, drivers, rng)
rides = make_rides(drivers, rng)

print(f"ресторанів: {len(restaurants)}, кур'єрів і водіїв: {len(drivers)}")
print(f"замовлень: {len(orders):_}, поїздок таксі: {len(rides):_}".replace("_", " "))
print(orders.head(3).to_string())

Майже 330 тисяч замовлень і 166 тисяч поїздок — достатньо, щоб різниця між підходами стала **видимою**, і досить мало, щоб усе вмістилося в пам'ять Colab.

**🔮 Прогноз:** що станеться з числами, якщо перезапустити ноутбук? А якщо змінити `42` на `7`?

<details>
<summary>Відповідь</summary>

Перезапуск дає ті самі числа: генератор із тим самим зерном видає ту саму послідовність. З іншим зерном кількість і суми зміняться, але закономірності (вихідні, популярні ресторани) лишаться — їх задає код, а не випадковість.

</details>

---
## 1. OLTP і OLAP: рядки проти стовпців

Бази даних розв'язують дві дуже різні задачі.

| | **OLTP** — транзакції | **OLAP** — аналітика |
|---|---|---|
| приклад | «створи замовлення № 330 001», «познач доставленим» | «виторг по районах за два місяці» |
| один запит торкається | кількох рядків | сотень тисяч чи мільйонів рядків |
| запитів | тисячі на секунду, від багатьох клієнтів | кілька на хвилину, від аналітика чи звіту |
| потрібно | швидко знайти й змінити **рядок**, не загубити жодної зміни | швидко прочитати **кілька стовпців** усієї таблиці |
| інструмент | PostgreSQL (урок 30), SQLite | DuckDB, ClickHouse, BigQuery, Snowflake |
| зберігання | **рядкове**: запис за записом | **стовпцеве**: стовпець за стовпцем |

Як лягають на диск ті самі три замовлення:

```text
рядкове сховище (PostgreSQL, SQLite):
  | 1, 412.50, delivered, Піца Поділ | 2, 610.00, delivered, Суші Оболонь | 3, 198.20, cancelled, Борщ Поділ |

стовпцеве сховище (DuckDB, Parquet):
  order_id : | 1 | 2 | 3 |
  total    : | 412.50 | 610.00 | 198.20 |
  status   : | delivered | delivered | cancelled |
  name     : | Піца Поділ | Суші Оболонь | Борщ Поділ |
```

Щоб порахувати `sum(total)`, рядкове сховище читає **всі** поля кожного запису. Стовпцеве — лише стовпець `total`: значення одного типу лежать поруч, їх легко стиснути й обробляти пачками.

### Приклад 1.1. Дві моделі в Python

Змоделюємо обидва способи на п'яти замовленнях: список кортежів (рядки) і словник списків (стовпці).

In [ ]:
COLUMNS = ["order_id", "restaurant_id", "driver_id", "items", "total", "status"]


def to_row_store(df):
    """Рядкове сховище: список записів, кожен запис — усі поля разом."""
    return [tuple(row) for row in df[COLUMNS].values.tolist()]


def to_column_store(df):
    """Стовпцеве сховище: для кожного стовпця — свій список значень."""
    return {column: df[column].tolist() for column in COLUMNS}

In [ ]:
sample = orders.head(5)
row_store = to_row_store(sample)
column_store = to_column_store(sample)

print("рядки:  ", row_store[:2], "...")
print("стовпці:", {k: v[:2] for k, v in list(column_store.items())[:3]}, "...")

total_index = COLUMNS.index("total")
read_rows = sum(len(row) for row in row_store)          # щоб дістати total, перебрали кожен запис повністю
read_columns = len(column_store["total"])               # тут — лише один список
print("сума з рядків:  ", round(sum(row[total_index] for row in row_store), 2), "| значень прочитано:", read_rows)
print("сума зі стовпця:", round(sum(column_store["total"]), 2), "| значень прочитано:", read_columns)

Сума однакова, але рядкова модель пройшла 30 значень, а стовпцева — 5. На 330 тисячах замовлень і 9 стовпцях різниця — у 9 разів менше даних з диска, і це ще без стиснення.

### Приклад 1.2. Рядкова база: SQLite

`sqlite3` — вбудований модуль Python: рядкова SQL-база в одному файлі чи в пам'яті, без сервера. Вона — наш «представник» PostgreSQL: обидві зберігають дані рядками. Завантажимо в неї замовлення й ресторани через `DataFrame.to_sql`.

In [ ]:
def load_sqlite(orders, restaurants):
    """Рядкова база в пам'яті з двома таблицями. Дати SQLite зберігає текстом."""
    lite = sqlite3.connect(":memory:")
    orders.assign(ts=orders["ts"].astype(str), order_date=orders["order_date"].astype(str)).to_sql("orders", lite, index=False)
    restaurants.to_sql("restaurants", lite, index=False)
    return lite


def best_time(fn, repeat=3):
    """Найкращий час із кількох запусків: менше залежить від випадкових пауз системи."""
    best = float("inf")
    for _ in range(repeat):
        start = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - start)
    return best

In [ ]:
lite = load_sqlite(orders, restaurants)
print("рядків у SQLite:", lite.execute("SELECT count(*) FROM orders").fetchone()[0])

### Приклад 1.3. Той самий GROUP BY у SQLite і DuckDB

Звіт власниці з уроку 30: для кожного району — кількість доставлених замовлень, виторг і середній чек. SQL **однаковий** для обох баз. Для DuckDB створимо базу в пам'яті `duckdb.connect()` і скопіюємо в неї ті самі DataFrame (`CREATE TABLE … AS SELECT * FROM orders` — DuckDB бачить DataFrame за іменем змінної, про це в розділі 2).

In [ ]:
REPORT_SQL = """
SELECT r.district,
       count(*)                AS orders,
       round(sum(o.total), 2)  AS revenue,
       round(avg(o.total), 2)  AS avg_check
FROM orders AS o
JOIN restaurants AS r USING (restaurant_id)
WHERE o.status = 'delivered'
GROUP BY r.district
ORDER BY revenue DESC
"""

In [ ]:
mem = duckdb.connect()                                   # DuckDB у пам'яті
mem.sql("CREATE TABLE orders AS SELECT * FROM orders")
mem.sql("CREATE TABLE restaurants AS SELECT * FROM restaurants")

lite_rows = lite.execute(REPORT_SQL).fetchall()
duck_rows = mem.sql(REPORT_SQL).fetchall()
assert lite_rows == duck_rows, "результати мають збігатися"

t_lite = best_time(lambda: lite.execute(REPORT_SQL).fetchall())
t_duck = best_time(lambda: mem.sql(REPORT_SQL).fetchall())
for row in duck_rows:
    print(row)
print(f"SQLite (рядки): {t_lite:.4f} с | DuckDB (стовпці): {t_duck:.4f} с | DuckDB швидший у {t_lite / t_duck:.0f} разів")

Результат однаковий до копійки, але DuckDB рахує в десятки разів швидше. Причини три: читає лише 4 потрібні стовпці з 9, обробляє значення **векторами** (пачками по 2048, а не по одному), і використовує всі ядра процесора.

**🔮 Прогноз:** чи завжди DuckDB швидший? Що буде, якщо робити 2000 окремих `INSERT` по одному рядку — як у диспетчерській, де замовлення приходять по одному?

<details>
<summary>Відповідь</summary>

Ні. Дрібні операції з одним рядком — сильна сторона рядкових баз. DuckDB на кожну вставку витрачає більше: дані треба розкласти по стовпцях і стиснути. Перевіримо в наступному прикладі.

</details>

### Приклад 1.4. Дрібні транзакції: тут виграє рядкова база

Імітуємо роботу OLTP: 2000 вставок по одному замовленню і 2000 пошуків за первинним ключем.

In [ ]:
def oltp_inserts(execute, n=2000):
    for i in range(n):
        execute("INSERT INTO tiny VALUES (?, ?)", (i, 100.0 + i))


def oltp_lookups(execute, n=2000):
    for i in range(n):
        execute("SELECT total FROM tiny WHERE id = ?", (i,)).fetchone()

In [ ]:
lite.execute("CREATE TABLE tiny (id INTEGER PRIMARY KEY, total REAL)")
mem.execute("CREATE TABLE tiny (id INTEGER PRIMARY KEY, total DOUBLE)")

timings = {}
for name, execute in [("SQLite", lite.execute), ("DuckDB", mem.execute)]:
    start = time.perf_counter()
    oltp_inserts(execute)
    inserts = time.perf_counter() - start
    start = time.perf_counter()
    oltp_lookups(execute)
    timings[name] = (inserts, time.perf_counter() - start)
    print(f"{name}: 2000 INSERT — {inserts:.3f} с, 2000 пошуків за id — {timings[name][1]:.3f} с")
print(f"на дрібних операціях SQLite швидший у {timings['DuckDB'][0] / timings['SQLite'][0]:.0f} разів")

Картина перевернулася: на дрібних операціях рядкова база швидша в сотні разів. Звідси правило архітектури:

- **замовлення, що відбуваються зараз** (OLTP) — у PostgreSQL;
- **аналітика над історією** (OLAP) — у стовпцевому рушії, як-от DuckDB.

Це не «краща» і «гірша» база, а два інструменти під два типи навантаження.

---
## 2. Основи DuckDB

DuckDB — **вбудована** (in-process) аналітична база: як SQLite, вона працює всередині твоєї Python-програми, без сервера, логіна й порту. `pip install duckdb` — і все. Але всередині — стовпцевий рушій для аналітики.

```text
PostgreSQL:   Python ──мережа──► сервер PostgreSQL ──► диск      (окремий процес, багато клієнтів)
DuckDB:       Python [ duckdb всередині процесу ] ──► файл .duckdb або Parquet   (один процес)
```

### Приклад 2.1. Перший запит

`duckdb.sql(...)` виконує SQL у базі за замовчуванням (у пам'яті) і повертає **relation** — об'єкт-результат. Його можна надрукувати, перетворити або використати в наступному запиті.

In [ ]:
first = duckdb.sql("SELECT 42 AS answer, 'Смачно + Таксі' AS service, current_date >= DATE '2026-01-01' AS fresh")

In [ ]:
print(type(first).__name__)
first

Relation **лінивий**: запит виконується, коли результат знадобився — під час друку, `.df()` чи `.fetchall()`. Останній рядок клітинки `first` Jupyter показує як таблицю з типами стовпців.

### Приклад 2.2. Запит до DataFrame за іменем змінної

Головна «магія» DuckDB у ноутбуках: у `FROM` можна написати **ім'я змінної** з DataFrame — DuckDB прочитає його напряму, без копіювання в базу (це називається replacement scan).

In [ ]:
STATUS_SQL = """
SELECT status, count(*) AS orders, round(avg(total), 2) AS avg_check
FROM orders                      -- це pandas DataFrame з розділу 0
GROUP BY status
ORDER BY orders DESC
"""

In [ ]:
duckdb.sql(STATUS_SQL)

### Приклад 2.3. Результат: `.df()`, `.fetchall()`, `.fetchone()`

Relation перетворюється на звичні об'єкти Python:

| метод | що повертає | коли брати |
|---|---|---|
| `.df()` | pandas DataFrame | далі аналіз чи графік у pandas |
| `.fetchall()` | список кортежів | як `cursor.fetchall()` у psycopg (урок 30) |
| `.fetchone()` | один кортеж | одне число: `count(*)`, `max(...)` |

In [ ]:
status_rel = duckdb.sql(STATUS_SQL)
status_df = status_rel.df()
status_rows = status_rel.fetchall()
total_orders = duckdb.sql("SELECT count(*) FROM orders").fetchone()[0]

In [ ]:
print(type(status_df).__name__, status_df.shape)
print(status_rows)
print("усього замовлень:", total_orders)
assert sum(row[1] for row in status_rows) == total_orders

### Приклад 2.4. База у файлі

`duckdb.connect("шлях.duckdb")` відкриває (або створює) базу в **одному файлі** — як SQLite. Таблиці переживають перезапуск програми. Створимо базу диспетчерської з чотирма таблицями: `CREATE TABLE … AS SELECT` (CTAS) створює таблицю одразу з даними, типи стовпців DuckDB бере з DataFrame. Гроші в DataFrame — `float`, а в базі їм місце в точному `DECIMAL(10, 2)` (як `numeric` в уроці 30): `SELECT * REPLACE (вираз AS стовпець)` бере всі стовпці, але один підміняє.

In [ ]:
DB_PATH = LAB / "smachno.duckdb"


MONEY = {"orders": "total", "rides": "fare"}       # стовпці з гривнями


def create_database(path):
    """Файл бази з чотирма таблицями з DataFrame. Повертає відкрите з'єднання."""
    con = duckdb.connect(str(path))
    for name, df in [("orders", orders), ("restaurants", restaurants), ("drivers", drivers), ("rides", rides)]:
        columns = f"* REPLACE ({MONEY[name]}::DECIMAL(10, 2) AS {MONEY[name]})" if name in MONEY else "*"
        con.sql(f"CREATE OR REPLACE TABLE {name} AS SELECT {columns} FROM df")
    return con

In [ ]:
con = create_database(DB_PATH)
con.close()                                     # закрили — дані лишилися у файлі

con = duckdb.connect(str(DB_PATH))              # відкрили знову
print(con.sql("SHOW TABLES").fetchall())
print(f"розмір файлу: {DB_PATH.stat().st_size / 1e6:.1f} МБ")
con.sql("DESCRIBE orders")

У `create_database` ім'я таблиці підставлено f-рядком, а дані — змінною `df`: імена таблиць і стовпців **не можна** передати параметром `?`, тому їх беремо лише з власного коду, ніколи від користувача. Зверни увагу на типи: `order_date` — `DATE`, `ts` — `TIMESTAMP`, `total` — `DECIMAL(10,2)`, цілі числа — `BIGINT`. З цього моменту `con` — наша основна база.

### Приклад 2.5. Похідні таблиці й подання

`CREATE TABLE … AS` зберігає **результат** запиту (копію даних), `CREATE VIEW` — лише **текст** запиту, який виконується щоразу заново. Зробимо подання «доставлені замовлення з районом» — далі воно заощадить нам `JOIN` у кожному запиті.

In [ ]:
DELIVERED_VIEW = """
CREATE OR REPLACE VIEW delivered AS
SELECT o.*, r.name AS restaurant, r.cuisine, r.district
FROM orders AS o
JOIN restaurants AS r USING (restaurant_id)
WHERE o.status = 'delivered'
"""
DAILY_TABLE = """
CREATE OR REPLACE TABLE daily AS
SELECT order_date, district, count(*) AS orders, round(sum(total), 2) AS revenue
FROM delivered
GROUP BY order_date, district
"""

In [ ]:
con.sql(DELIVERED_VIEW)
con.sql(DAILY_TABLE)
print("доставлених:", con.sql("SELECT count(*) FROM delivered").fetchone()[0])
print("рядків у daily (61 день × 6 районів):", con.sql("SELECT count(*) FROM daily").fetchone()[0])

### Приклад 2.6. Параметри `?`, а не f-рядки

Так само, як у psycopg (урок 30, розділ «Параметри, а не f-рядки»), значення від користувача передаємо **параметрами**. У DuckDB місце для значення — `?` (або `$name`), значення — списком другим аргументом. f-рядок відкриває двері SQL-ін'єкції.

In [ ]:
def restaurant_orders_unsafe(con, name):
    """❌ Текст користувача стає частиною SQL."""
    return con.sql(f"SELECT count(*) FROM delivered WHERE restaurant = '{name}'").fetchone()[0]


def restaurant_orders(con, name):
    """✅ Значення йде окремо від SQL — базі все одно, які в ньому лапки."""
    return con.execute("SELECT count(*) FROM delivered WHERE restaurant = ?", [name]).fetchone()[0]

In [ ]:
attack = "x' OR '1'='1"
print("чесний запит:     ", restaurant_orders(con, "Суші Оболонь"))
print("ін'єкція, f-рядок:", restaurant_orders_unsafe(con, attack), "← повернула ВСІ замовлення")
print("ін'єкція, ?:      ", restaurant_orders(con, attack), "← такого ресторану немає")

`con.execute(sql, params)` повертає саме з'єднання (як курсор); `con.sql(sql, params=[...])` — relation. Обидва приймають параметри.

### Приклад 2.7. `GROUP BY ALL` і `ORDER BY ALL`

У звичайному SQL кожен неагрегований стовпець треба повторити в `GROUP BY` (урок 30, «Кожен стовпець — або в GROUP BY, або в агрегаті»). DuckDB уміє `GROUP BY ALL`: групувати за **всіма** стовпцями `SELECT`, що не є агрегатами. Ще кілька зручностей DuckDB: `ORDER BY ALL`, `SELECT * EXCLUDE (...)`, `FROM таблиця` без `SELECT *`.

In [ ]:
CLASSIC_SQL = """
SELECT district, cuisine, count(*) AS orders
FROM delivered
GROUP BY district, cuisine
ORDER BY district, cuisine
"""
FRIENDLY_SQL = """
SELECT district, cuisine, count(*) AS orders
FROM delivered
GROUP BY ALL
ORDER BY ALL
"""

In [ ]:
assert con.sql(CLASSIC_SQL).fetchall() == con.sql(FRIENDLY_SQL).fetchall()
print(con.sql(FRIENDLY_SQL).limit(4))
print(con.sql("FROM drivers SELECT * EXCLUDE (vehicle) LIMIT 3"))

### Приклад 2.8. `SUMMARIZE`: перший погляд на таблицю

`SUMMARIZE таблиця` — як `df.describe()` у pandas, але для всіх стовпців одразу: мінімум, максимум, приблизна кількість унікальних, середнє, квартилі, частка `NULL`. Перше, що роблять з незнайомими даними.

In [ ]:
SUMMARY_SQL = """
SELECT column_name, column_type, min, max, approx_unique, null_percentage
FROM (SUMMARIZE orders)
"""

In [ ]:
con.sql(SUMMARY_SQL)

`approx_unique` — **приблизна** оцінка (алгоритм HyperLogLog): точний підрахунок унікальних на мільйонах рядків дорогий, а для огляду досить «близько 40 тисяч клієнтів».

### Приклад 2.9. `PIVOT`: рядки → стовпці

Звіт «кількість замовлень: райони в рядках, кухні в стовпцях». У pandas це `pivot_table` (бонус Pandas), у DuckDB — `PIVOT … ON … USING …`.

In [ ]:
PIVOT_SQL = """
PIVOT delivered
ON cuisine
USING count(*)
GROUP BY district
ORDER BY district
"""

In [ ]:
con.sql(PIVOT_SQL)

Значення стовпця `cuisine` стали **іменами стовпців**. Зворотна операція — `UNPIVOT` (у pandas — `melt`).

### Приклад 2.10. `EXPLAIN` і `EXPLAIN ANALYZE`

`EXPLAIN` показує **план** — як DuckDB збирається виконати запит (як у PostgreSQL, урок 30, розділ «Індекси»). `EXPLAIN ANALYZE` виконує запит і додає реальний час і кількість рядків на кожному кроці. Читається знизу вгору: сканування → фільтр → групування.

In [ ]:
def explain(con, sql, analyze=False):
    """Надрукувати план запиту (текстом, як у консолі DuckDB)."""
    prefix = "EXPLAIN ANALYZE " if analyze else "EXPLAIN "
    print(con.sql(prefix + sql).fetchall()[0][1])

In [ ]:
explain(con, "SELECT district, sum(total) FROM delivered GROUP BY district")

Знизу — `SEQ_SCAN` двох таблиць (`orders` і `restaurants`), причому з `orders` читаються лише потрібні стовпці (**Projections**), а фільтр `status = 'delivered'` виконано прямо під час сканування (**Filters**). Далі `HASH_JOIN`, `HASH_GROUP_BY`. Тепер `ANALYZE`: ті самі вузли, але з реальним часом.

In [ ]:
explain(con, "SELECT district, sum(total) FROM delivered GROUP BY district", analyze=True)

---
## 3. Віконні функції

`GROUP BY` **згортає** рядки: з 366 рядків `daily` залишилося б 6 (по району). **Віконна функція** рахує по групі, але **не згортає**: кожен рядок лишається на місці й отримує додатковий стовпець — накопичену суму, місце в рейтингу, значення попереднього дня.

```text
функція(...) OVER (PARTITION BY district   -- окреме «вікно» для кожного району
                   ORDER BY order_date)    -- порядок усередині вікна
```

### Приклад 3.1. Накопичений виторг

«Скільки Поділ заробив від 1 вересня до кожного дня включно?» — `sum(revenue) OVER (PARTITION BY district ORDER BY order_date)`. З `ORDER BY` сума рахується від початку вікна до поточного рядка.

In [ ]:
RUNNING_SQL = """
SELECT order_date, district, revenue,
       sum(revenue) OVER (PARTITION BY district ORDER BY order_date) AS running_revenue
FROM daily
WHERE district = 'Поділ'
ORDER BY order_date
LIMIT 5
"""

In [ ]:
con.sql(RUNNING_SQL)

Кожен рядок `running_revenue` = попередній + `revenue` цього дня. Стовпці `order_date` і `revenue` лишилися — рядки не згорнулися.

### Приклад 3.2. Рейтинг ресторанів у районі

`rank() OVER (PARTITION BY district ORDER BY revenue DESC)` нумерує ресторани **всередині кожного району** від найприбутковішого. Щоб лишити лише двох лідерів, класичний SQL вимагає підзапиту: у `WHERE` віконні функції ще не пораховані (порядок виконання `SELECT` — урок 30).

In [ ]:
RANK_SUBQUERY_SQL = """
WITH ranked AS (
    SELECT district, restaurant, round(sum(total), 2) AS revenue,
           rank() OVER (PARTITION BY district ORDER BY sum(total) DESC) AS place
    FROM delivered
    GROUP BY district, restaurant
)
SELECT * FROM ranked
WHERE place <= 2
ORDER BY district, place
"""

In [ ]:
con.sql(RANK_SUBQUERY_SQL)

Віконна функція працює **після** `GROUP BY`, тому всередині `OVER` можна писати агрегат `sum(total)`.

### Приклад 3.3. `QUALIFY`: фільтр за віконною функцією

`QUALIFY` — це `WHERE` для віконних функцій: виконується після них. Той самий звіт без підзапиту:

In [ ]:
QUALIFY_SQL = """
SELECT district, restaurant, round(sum(total), 2) AS revenue,
       rank() OVER (PARTITION BY district ORDER BY sum(total) DESC) AS place
FROM delivered
GROUP BY district, restaurant
QUALIFY place <= 2
ORDER BY district, place
"""

In [ ]:
assert con.sql(QUALIFY_SQL).fetchall() == con.sql(RANK_SUBQUERY_SQL).fetchall(), "QUALIFY має дати те саме, що підзапит"
con.sql(QUALIFY_SQL)

Порядок виконання з `QUALIFY`: FROM → WHERE → GROUP BY → HAVING → **віконні функції** → **QUALIFY** → SELECT → ORDER BY → LIMIT. `QUALIFY` є в DuckDB, Snowflake, BigQuery; у PostgreSQL його немає — там лишається підзапит. І ще одна деталь: разом із `GROUP BY ALL` DuckDB 1.5 `QUALIFY` поки не приймає (`Binder Error: Combining QUALIFY with GROUP BY ALL is not supported yet`) — тому стовпці групування тут перелічено явно.

| функція | що дає | при однакових значеннях |
|---|---|---|
| `row_number()` | 1, 2, 3, 4 | різні номери, порядок довільний |
| `rank()` | 1, 2, 2, 4 | однакове місце, далі пропуск |
| `dense_rank()` | 1, 2, 2, 3 | однакове місце, без пропуску |

### Приклад 3.4. `lag`: зміна день до дня

`lag(revenue)` дає значення з **попереднього** рядка вікна. Різниця з поточним — зростання чи падіння виторгу за день. День тижня — зі списку назв: `isodow` дає 1 для понеділка … 7 для неділі, а списки в DuckDB нумеруються з 1.

In [ ]:
LAG_SQL = """
SELECT order_date, ['пн', 'вт', 'ср', 'чт', 'пт', 'сб', 'нд'][isodow(order_date)] AS weekday, revenue,
       revenue - lag(revenue) OVER (PARTITION BY district ORDER BY order_date) AS change
FROM daily
WHERE district = 'Поділ'
ORDER BY order_date
LIMIT 8
"""

In [ ]:
con.sql(LAG_SQL)

У першого дня попереднього немає — `lag` дає `NULL`, і різниця теж `NULL`. Добре видно вихідні: у суботу виторг стрибає вгору, у понеділок падає.

### Задача 1. Три найкращі кур'єри кожного району

Власниця хоче премію для **трьох** найкращих кур'єрів у **кожному районі** (район — за рестораном замовлення). Допиши `top_drivers(con, n)`:

- рахує для кожної пари «район — кур'єр» кількість доставлених замовлень `orders` і виторг `revenue` (`round(sum(total), 2)`), бере ім'я кур'єра з таблиці `drivers` (стовпець `driver`);
- місце `place` — `rank()` усередині району за виторгом, від більшого;
- лишає перші `n` місць через **`QUALIFY`**, а `n` передає **параметром `?`**, не f-рядком;
- повертає DataFrame зі стовпцями `district, driver, orders, revenue, place`, відсортований за районом і місцем.

Підказка: почни з подання `delivered` і `JOIN drivers USING (driver_id)`. Перевірка — у наступній клітинці: вона порівнює з тим самим звітом, порахованим у pandas.

In [ ]:
def top_drivers(con, n=3):
    # YOUR CODE HERE
    pass


In [ ]:
import inspect

result = top_drivers(con, 3)
assert isinstance(result, pd.DataFrame), "top_drivers має повертати DataFrame (метод .df())"
assert list(result.columns) == ["district", "driver", "orders", "revenue", "place"], f"стовпці: {list(result.columns)}"
assert len(result) == 18, f"6 районів × 3 місця = 18 рядків, а маємо {len(result)}"
assert len(top_drivers(con, 1)) == 6, "з n=1 — по одному кур'єру на район"

check = (orders[orders["status"] == "delivered"]
         .merge(restaurants, on="restaurant_id").merge(drivers, on="driver_id", suffixes=("", "_driver"))
         .groupby(["district", "name_driver"], as_index=False)["total"].sum())
check["place"] = check.groupby("district")["total"].rank(ascending=False, method="min")
check = check[check["place"] <= 3].sort_values(["district", "place"], ignore_index=True)
assert result["driver"].tolist() == check["name_driver"].tolist(), "імена кур'єрів не збігаються з перевіркою в pandas"
assert np.allclose(result["revenue"], check["total"].round(2)), "виторг не збігається з перевіркою в pandas"

source = inspect.getsource(top_drivers)
assert "QUALIFY" in source.upper(), "використай QUALIFY, а не підзапит"
assert "?" in source and "{n}" not in source, "n передай параметром ?, не f-рядком"
print(result.head(6).to_string(index=False))
print("✅ Задача 1 розв'язана")

---
## 4. pandas і DuckDB

У бонусі Pandas усі звіти були на pandas. DuckDB не заміняє pandas, а **доповнює**: важке групування й з'єднання — SQL у DuckDB, графік і дрібна обробка — pandas. Дані між ними ходять без копіювання.

### Приклад 4.1. Той самий звіт двома способами

Звіт по районах з прикладу 1.3: pandas (`merge` → `groupby` → `agg`) і DuckDB прямо над тими самими DataFrame.

In [ ]:
def report_pandas(orders, restaurants):
    delivered = orders[orders["status"] == "delivered"].merge(restaurants, on="restaurant_id")
    return (delivered.groupby("district")
            .agg(orders=("order_id", "count"), revenue=("total", "sum"), avg_check=("total", "mean"))
            .round(2).sort_values("revenue", ascending=False).reset_index())


def report_duckdb(orders, restaurants):
    return duckdb.sql(REPORT_SQL).df()       # orders і restaurants у SQL — ці самі DataFrame

In [ ]:
by_pandas = report_pandas(orders, restaurants)
by_duckdb = report_duckdb(orders, restaurants)
assert by_pandas["district"].tolist() == by_duckdb["district"].tolist()
assert np.allclose(by_pandas["revenue"], by_duckdb["revenue"])

t_pandas = best_time(lambda: report_pandas(orders, restaurants))
t_duckdb = best_time(lambda: report_duckdb(orders, restaurants))
print(by_duckdb.to_string(index=False))
print(f"pandas: {t_pandas:.4f} с | DuckDB над тими самими DataFrame: {t_duckdb:.4f} с")

Числа однакові, а час на 330 тисячах рядків — майже однаковий: pandas теж рахує векторно (через NumPy). Різниця з'являється, коли даних більше: pandas будує проміжний DataFrame на кожному кроці (результат `merge` — ще 300 тисяч рядків у пам'яті) і рахує одним ядром, а DuckDB виконує весь запит за один прохід кількома ядрами і вміє читати дані, більші за пам'ять. На таких обсягах обирай те, що читається легше.

| задача | зручніше |
|---|---|
| з'єднання кількох великих таблиць, групування, віконні функції | DuckDB (SQL) |
| дані більші за пам'ять, файли Parquet / CSV на диску | DuckDB — читає файли частинами |
| графік, дрібне перетворення, машинне навчання | pandas |
| звіт, який має читати аналітик, що знає SQL | DuckDB |

### Приклад 4.2. Arrow: обмін без копіювання

**Apache Arrow** — стандартний формат стовпцевих даних **у пам'яті**. DuckDB, pandas (з типами `pyarrow`), Polars і Parquet-бібліотеки розуміють його однаково, тож таблицю можна передати з однієї бібліотеки в іншу без перетворення рядок за рядком. `.to_arrow_table()` дає таблицю `pyarrow.Table`, а DuckDB так само, за іменем змінної, читає Arrow-таблиці.

In [ ]:
rides_arrow = con.sql("SELECT ride_id, ride_date, driver_id, distance_km, fare FROM rides").to_arrow_table()

In [ ]:
print(type(rides_arrow).__name__, rides_arrow.num_rows, "рядків")
print(rides_arrow.schema)
print(duckdb.sql("SELECT round(avg(fare), 2) AS avg_fare, round(avg(distance_km), 1) AS avg_km FROM rides_arrow"))
rides_back = rides_arrow.to_pandas()
print("назад у pandas:", rides_back.shape)

У схемі Arrow `ride_date` — `date32[day]`, тип зберігся. Arrow — «спільна мова» стовпцевих інструментів; Parquet — те саме, але **на диску**. Про нього — далі.

---
## 5. Формати файлів: CSV, Parquet, JSON

Аналітичні дані часто живуть не в базі, а у **файлах**: так їх дешево зберігати, легко передати іншій команді й прочитати будь-яким інструментом. DuckDB читає й пише CSV, JSON і Parquet напряму SQL-запитом.

| | CSV | JSON (по рядку на запис) | Parquet |
|---|---|---|---|
| вигляд | текст, рядок = запис | текст, рядок = об'єкт | двійковий, **стовпцевий** |
| типи | немає — усе текст, типи вгадують | числа, рядки, `null`; дат немає | збережені: `DATE`, `TIMESTAMP`, `DECIMAL`… |
| стиснення | немає (або gzip усього файлу) | немає | кожен стовпець окремо (Snappy, ZSTD) |
| прочитати 2 стовпці з 9 | читаємо весь файл | читаємо весь файл | читаємо лише 2 стовпці |
| відкрити в Excel / блокноті | так | так | ні |
| для чого | обмін з людьми і старими системами | події, API, логи | аналітика, data lake |

### Приклад 5.1. Пишемо CSV і Parquet

`COPY таблиця TO 'файл' (FORMAT …)` записує результат у файл. Для Parquet задамо розмір **групи рядків** (row group) — 20 000 рядків: файл ділиться на частини, і для кожної зберігаються мінімум і максимум кожного стовпця. Знадобиться в прикладі 5.4.

In [ ]:
FILES = LAB / "files"
FILES.mkdir(exist_ok=True)
CSV_PATH = FILES / "orders.csv"
PARQUET_PATH = FILES / "orders.parquet"
ZSTD_PATH = FILES / "orders_zstd.parquet"


def write_formats(con):
    con.sql(f"COPY orders TO '{CSV_PATH}' (HEADER)")
    con.sql(f"COPY orders TO '{PARQUET_PATH}' (FORMAT PARQUET, ROW_GROUP_SIZE 20000)")
    con.sql(f"COPY orders TO '{ZSTD_PATH}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 20000)")

In [ ]:
write_formats(con)
for path in [CSV_PATH, PARQUET_PATH, ZSTD_PATH]:
    print(f"{path.name:22} {path.stat().st_size / 1e6:6.1f} МБ")
print(f"Parquet менший за CSV у {CSV_PATH.stat().st_size / PARQUET_PATH.stat().st_size:.1f} раза")

Parquet у кілька разів менший: стовпець `status` з трьома значеннями зберігається **словником** (0, 1, 2 замість слова), відсортовані `order_id` і `ts` — різницями сусідніх значень, і все це ще стискається. ZSTD стискає сильніше за типовий Snappy, але трохи повільніше.

Шлях до файлу тут підставлено f-рядком: це **наш** шлях з коду, а не текст користувача. Для значень від користувача — лише `?`.

### Приклад 5.2. Читання: той самий запит над CSV і Parquet

DuckDB читає файл прямо в `FROM`: `read_csv('...')`, `read_parquet('...')` або просто `'шлях.parquet'` у лапках.

In [ ]:
FILE_REPORT = """
SELECT status, count(*) AS orders, round(sum(total), 2)::DECIMAL(14, 2) AS revenue   -- однаковий тип для CSV і Parquet
FROM {source}
GROUP BY ALL
ORDER BY ALL
"""


def report_from(con, source):
    return con.sql(FILE_REPORT.format(source=source)).fetchall()

In [ ]:
csv_source, parquet_source = f"read_csv('{CSV_PATH}')", f"read_parquet('{PARQUET_PATH}')"
assert report_from(con, csv_source) == report_from(con, parquet_source)
t_csv = best_time(lambda: report_from(con, csv_source))
t_parquet = best_time(lambda: report_from(con, parquet_source))
for status, n, revenue in report_from(con, parquet_source):
    print(f"{status:10} {n:7} {revenue:>13} грн")
print(f"CSV: {t_csv:.3f} с | Parquet: {t_parquet:.4f} с | Parquet швидший у {t_csv / t_parquet:.0f} разів")

CSV треба **розібрати** — кожен символ кожного рядка, навіть якщо потрібні два стовпці, і ще вгадати типи. Parquet віддає два стовпці вже з типами. У pandas так само: `pd.read_parquet(path, columns=["status", "total"])` читає з диска лише ці стовпці, а `pd.read_csv(path, usecols=...)` однаково проходить увесь файл.

### Приклад 5.3. Що всередині Parquet

Parquet зберігає **схему** (імена й типи стовпців) і **метадані** груп рядків. DuckDB показує їх функціями `parquet_schema` і `parquet_metadata`.

In [ ]:
SCHEMA_SQL = f"SELECT column_name, column_type FROM (DESCRIBE SELECT * FROM read_parquet('{PARQUET_PATH}'))"
ROW_GROUPS_SQL = f"""
SELECT row_group_id, row_group_num_rows AS rows, stats_min AS min_date, stats_max AS max_date
FROM parquet_metadata('{PARQUET_PATH}')
WHERE path_in_schema = 'order_date'
ORDER BY row_group_id
"""

In [ ]:
print(con.sql(SCHEMA_SQL))
print(con.sql(ROW_GROUPS_SQL).limit(5))
print("груп рядків:", len(con.sql(ROW_GROUPS_SQL).fetchall()))

Типи збереглися всі, навіть `DECIMAL(10,2)`. Для кожної групи рядків (DuckDB округлює розмір до 20 480 — кратного своєму вектору 2048) відомий діапазон дат: група 0 — 1–5 вересня, група 1 — 5–8 вересня… Дані відсортовані за часом, тому сусідні групи ділять щонайбільше один межовий день.

### Приклад 5.4. Projection pushdown і predicate pushdown

- **Projection pushdown** — «читай лише потрібні стовпці»: вузол сканування в плані має рядок `Projections`.
- **Predicate pushdown** — «фільтруй під час читання»: умова `WHERE` спускається в сканування (`Filters`), а групи рядків, де `min`/`max` не підходять, **пропускаються** зовсім.

In [ ]:
ONE_DAY_SQL = f"""
SELECT count(*) AS orders, round(sum(total), 2) AS revenue
FROM read_parquet('{PARQUET_PATH}')
WHERE order_date = DATE '2026-10-15'
"""

In [ ]:
explain(con, ONE_DAY_SQL)

In [ ]:
all_groups = con.sql(f"SELECT count(*) FROM ({ROW_GROUPS_SQL})").fetchone()[0]
groups = con.sql(f"SELECT count(*) FROM ({ROW_GROUPS_SQL}) WHERE min_date <= '2026-10-15' AND max_date >= '2026-10-15'").fetchone()[0]
day_orders, day_revenue = con.sql(ONE_DAY_SQL).fetchone()
print(f"15 жовтня: {day_orders} замовлень, {day_revenue} грн | груп рядків, де може бути цей день: {groups} з {all_groups}")

У плані — `Projections: total`: з 9 стовпців у запит іде лише один (`order_date` потрібен тільки фільтру, який виконується всередині сканування — `Filters: order_date='2026-10-15'`). За метаданими 15 жовтня може бути лише в одній групі рядків з 17 — решту DuckDB навіть не читає з диска. Тому дані, які часто фільтрують за датою, варто **сортувати за датою** перед записом у Parquet.

### Приклад 5.5. CSV: типи вгадуються

У CSV типів немає. `read_csv` (або `read_csv_auto` — те саме) читає перші рядки й **вгадує** роздільник, заголовок і типи (sniffing). `DESCRIBE` показує, що вийшло.

In [ ]:
SNIFF_SQL = f"DESCRIBE SELECT * FROM read_csv_auto('{CSV_PATH}')"

In [ ]:
con.sql(SNIFF_SQL).select("column_name, column_type")

Майже все вгадано, бо файл записав сам DuckDB. Але `total` у базі був `DECIMAL(10,2)`, а з CSV повернувся `DOUBLE`: у тексті `412.50` — просто число з крапкою, точний грошовий тип загубився. З чужими файлами буває гірше: `"0501234567"` стане числом і втратить нуль, дата `03.10.2026` лишиться текстом. Тоді типи задають явно: `read_csv('f.csv', types={'phone': 'VARCHAR'})`, або читають усе текстом (`all_varchar = true`) і перетворюють самі — так зробимо в розділі 7.

### Приклад 5.6. JSON

Події від мобільних застосунків і API зазвичай приходять у **JSON**, по одному об'єкту на рядок (NDJSON, JSON Lines). DuckDB пише його `COPY … (FORMAT JSON)` і читає `read_json_auto`.

In [ ]:
JSON_PATH = FILES / "rides.json"


def write_rides_json(con):
    con.sql(f"COPY (SELECT * FROM rides LIMIT 1000) TO '{JSON_PATH}' (FORMAT JSON)")

In [ ]:
write_rides_json(con)
with open(JSON_PATH, encoding="utf-8") as f:
    print(f.readline().strip())
print(con.sql(f"DESCRIBE SELECT * FROM read_json_auto('{JSON_PATH}')").select("column_name, column_type"))

JSON зберігає імена ключів у кожному рядку — тому на тих самих даних він найбільший з трьох форматів. Типи DuckDB вгадав, навіть `DATE` і `TIMESTAMP` за виглядом рядка `"2026-09-01"`, але `fare`, як і в CSV, став `DOUBLE`.

### Приклад 5.7. Багато файлів — одна таблиця

Шаблон `*` у шляху читає **всі** файли, що підходять, як одну таблицю. `filename = true` додає стовпець з іменем файлу, з якого прийшов рядок. Запишемо поїздки по місяцях у два файли і прочитаємо разом.

In [ ]:
def write_rides_by_month(con):
    for month in (9, 10):
        con.execute(f"COPY (SELECT * FROM rides WHERE month(ride_date) = {month}) "
                    f"TO '{FILES / f'rides_2026_{month:02}.parquet'}' (FORMAT PARQUET)")


GLOB_SQL = f"""
SELECT parse_filename(filename) AS file, count(*) AS rides, round(sum(fare), 2) AS revenue
FROM read_parquet('{FILES}/rides_*.parquet', filename = true)
GROUP BY ALL
ORDER BY ALL
"""

In [ ]:
write_rides_by_month(con)
con.sql(GLOB_SQL)

Це вже крок до озера даних: **таблиця — це папка з файлами**, а не об'єкт усередині бази.

---
## 6. Data lake: озеро даних

| | **data warehouse** (сховище) | **data lake** (озеро) | **lakehouse** |
|---|---|---|---|
| що зберігає | очищені таблиці з суворою схемою | будь-які файли: сирі й чисті, CSV, JSON, Parquet, картинки | файли в озері + шар таблиць поверх них |
| схема | **schema-on-write**: перевірка під час запису | **schema-on-read**: схему застосовує той, хто читає | схема таблиці в метаданих, файли — Parquet |
| де лежать дані | усередині СКБД | папки на диску чи в об'єктному сховищі (S3) | об'єктне сховище |
| сильна сторона | надійні SQL-звіти | дешево, будь-який формат, «зберегти все» | обидва світи: дешеве сховище і транзакції |
| ризик | дорого, пізно додавати нові джерела | «болото даних»: ніхто не знає, що де й чи можна вірити | складніша інфраструктура |
| приклади | PostgreSQL як сховище, Snowflake, BigQuery | папки Parquet + DuckDB / Spark | Apache Iceberg, Delta Lake, DuckLake |

### Приклад 6.1. Schema-on-write проти schema-on-read

**Schema-on-write** — як у PostgreSQL з уроку 30: таблиця з типами й обмеженнями не пустить поганий рядок. **Schema-on-read** — файл приймає що завгодно, а перевіряти доводиться під час читання.

In [ ]:
RAW_CSV = FILES / "raw_sample.csv"
RAW_TEXT = """order_id,total,status
1,412.50,delivered
2,"610,00",Delivered
3,n/a,delivered
4,-50,cancelled
"""
STRICT_TABLE = """
CREATE OR REPLACE TABLE strict_orders (
    order_id INTEGER PRIMARY KEY,
    total    DECIMAL(10, 2) NOT NULL CHECK (total > 0),
    status   VARCHAR CHECK (status IN ('delivered', 'cancelled', 'returned'))
)
"""

In [ ]:
con.sql(STRICT_TABLE)
con.execute("INSERT INTO strict_orders VALUES (?, ?, ?)", [1, 412.50, "delivered"])
try:
    con.execute("INSERT INTO strict_orders VALUES (?, ?, ?)", [4, -50, "cancelled"])
except duckdb.ConstraintException as error:
    print("schema-on-write:", str(error).splitlines()[0])

RAW_CSV.write_text(RAW_TEXT, encoding="utf-8")       # файлу байдуже, що в ньому
print("schema-on-read, усе як текст:")
print(con.sql(f"SELECT * FROM read_csv('{RAW_CSV}', all_varchar = true)"))
print(con.sql(f"SELECT order_id, total, TRY_CAST(total AS DECIMAL(10, 2)) AS total_typed FROM read_csv('{RAW_CSV}', all_varchar = true)"))

База відмовилась записати `-50` одразу. Файл записав усе — і `"610,00"` з комою, і `n/a`, і `Delivered` з великої літери. `TRY_CAST` перетворює, а там, де не вийшло, дає `NULL` замість помилки (звичайний `CAST` упав би на першому ж `n/a`). Озеро зберігає все, тому **очищення — окремий крок** конвеєра.

### Приклад 6.2. Hive-розбиття: папка на кожне значення

**Розбиття** (partitioning) — дані однієї таблиці розкладені по папках за значенням стовпця. Стиль Hive: ім'я папки — `стовпець=значення`. `COPY … (PARTITION_BY (...))` розкладає сам.

```text
lake/orders_by_month/
├── year=2026/
│   ├── month=9/
│   │   ├── status=cancelled/data_0.parquet
│   │   ├── status=delivered/data_0.parquet
│   │   └── status=returned/data_0.parquet
│   └── month=10/ ...
```

In [ ]:
LAKE = LAB / "lake"
BY_MONTH = LAKE / "orders_by_month"
LAKE.mkdir(exist_ok=True)


def write_partitioned(con):
    con.sql(f"""
        COPY (SELECT *, year(order_date) AS year, month(order_date) AS month FROM orders)
        TO '{BY_MONTH}' (FORMAT PARQUET, PARTITION_BY (year, month, status), OVERWRITE_OR_IGNORE)
    """)


def tree(root, limit=12):
    """Надрукувати файли під root: шлях відносно root і розмір. Лише перші limit файлів."""
    files = sorted(p for p in Path(root).rglob("*") if p.is_file())
    for path in files[:limit]:
        print(f"  {path.relative_to(root).as_posix():55} {path.stat().st_size / 1e3:8.1f} КБ")
    if len(files) > limit:
        print(f"  ... ще {len(files) - limit}")
    size = sum(p.stat().st_size for p in files)
    print(f"  файлів: {len(files)}, разом " + (f"{size / 1e6:.1f} МБ" if size >= 1e6 else f"{size / 1e3:.1f} КБ"))

In [ ]:
write_partitioned(con)
tree(BY_MONTH)

У самих файлах стовпців `year`, `month`, `status` **немає** — їхні значення записані в іменах папок.

!!! warning "Ключі розбиття — латиницею або числами"
    Значення ключа стає ім'ям папки. Кирилицю DuckDB кодує як `district=%D0%9F%D0%BE…` — читати таке озеро вміє, але людині незручно. Тому розбивають за датою, роком, місяцем, кодом чи `id`.

### Приклад 6.3. Partition pruning: читаємо лише потрібні папки

`read_parquet(..., hive_partitioning = true)` повертає ключі з імен папок як звичайні стовпці. Якщо фільтр стосується ключа розбиття, DuckDB відкидає зайві папки ще **до** читання — це **partition pruning**.

In [ ]:
PRUNING_SQL = f"""
SELECT count(*) AS orders, round(sum(total), 2) AS revenue
FROM read_parquet('{BY_MONTH}/**/*.parquet', hive_partitioning = true)
WHERE year = 2026 AND month = 10 AND status = 'delivered'
"""

In [ ]:
pruned_orders, pruned_revenue = con.sql(PRUNING_SQL).fetchone()
print(f"доставлено в жовтні: {pruned_orders} замовлень на {pruned_revenue} грн")
explain(con, PRUNING_SQL)

У плані — `File Filters` і `Scanning Files: 1/6`: з шести файлів прочитано один. На озері за кілька років різниця — між секундами й годинами.

### Приклад 6.4. Проблема дрібних файлів

Якщо розбити надто дрібно, файлів стає забагато: кожен треба відкрити, прочитати його метадані, а в хмарному сховищі — ще й зробити окремий мережевий запит. Розіб'ємо ті самі замовлення за **рестораном і днем**: 36 × 61 = 2196 файлів по ~150 рядків.

In [ ]:
TINY = LAKE / "orders_tiny_files"


def write_tiny_files(con):
    con.sql(f"""
        COPY orders TO '{TINY}'
        (FORMAT PARQUET, PARTITION_BY (restaurant_id, order_date), OVERWRITE_OR_IGNORE)
    """)


def compare_small_files(con):
    tiny_sql = f"SELECT status, sum(total) FROM read_parquet('{TINY}/**/*.parquet', hive_partitioning = true) GROUP BY ALL"
    one_sql = f"SELECT status, sum(total) FROM read_parquet('{PARQUET_PATH}') GROUP BY ALL"
    return best_time(lambda: con.sql(tiny_sql).fetchall()), best_time(lambda: con.sql(one_sql).fetchall())

In [ ]:
write_tiny_files(con)
partitions = sum(1 for p in TINY.glob("*/*") if p.is_dir())
tiny_files = list(TINY.rglob("*.parquet"))
print(f"розділів: {partitions}, файлів: {len(tiny_files)}, "
      f"разом {sum(p.stat().st_size for p in tiny_files) / 1e6:.1f} МБ проти {PARQUET_PATH.stat().st_size / 1e6:.1f} МБ одного файлу")
t_tiny, t_one = compare_small_files(con)
print(f"дрібні файли: {t_tiny:.3f} с | один файл: {t_one:.4f} с | у {t_tiny / t_one:.0f} разів повільніше")

Файлів трохи більше, ніж розділів: DuckDB пише в кілька потоків, і інколи в один розділ потрапляють два файли (`data_0`, `data_1`) — тому точне число в тебе може відрізнятися. Ті самі дані займають більше місця (у кожному файлі — своя схема й метадані), а читаються в десятки разів повільніше. Правила:

- файл Parquet — від десятків до сотень мегабайтів; групи рядків — десятки-сотні тисяч рядків;
- розбивати за ключем, за яким **справді фільтрують** (дата, місяць), і так, щоб у розділі були хоча б мегабайти;
- якщо дрібні файли вже накопичились (наприклад, кожні 5 хвилин приходить новий) — періодично **ущільнювати** (compaction): переписувати розділ в один файл.

---
## 7. Медальйонна архітектура: bronze → silver → gold

Найпоширеніший спосіб навести лад в озері — **три шари**, як медалі:

```text
 джерела                 BRONZE                   SILVER                     GOLD
 застосунок ──CSV──►  сирі файли як є    ──►  очищені, типізовані,   ──►  вітрини для звітів:
 таксі      ──JSON─►  (повтори, сміття)       без повторів, Parquet,      агрегати, зіркова схема
                      нічого не змінюємо      розбиті за днем             (розділ 8)
                            │
                            └── відкинуті рядки ──► QUARANTINE (з причиною)
```

| шар | що там | хто читає | правило |
|---|---|---|---|
| **bronze** | копія джерела байт у байт | конвеєр, розслідування помилок | лише дописувати, ніколи не правити |
| **silver** | ті самі події, але чисті: типи, без повторів, перевірені | аналітики, data science | одна подія — один рядок |
| **gold** | агрегати й вітрини під конкретні питання | дашборди, власниця | маленькі, швидкі, зрозумілі |

Bronze зберігає **все**, тому будь-яку помилку очищення можна виправити й перерахувати silver і gold заново.

**ETL чи ELT?** Класичний ETL (extract → transform → load) очищує дані **до** завантаження у сховище. В озері частіше **ELT**: спершу завантажити сире (bronze), а перетворювати потім, усередині аналітичного рушія (SQL у DuckDB). Наш конвеєр — саме такий.

### Приклад 7.1. Bronze: сирі журнали застосунку

Мобільний застосунок щодня вивантажує журнал замовлень у CSV, а сервіс таксі — поїздки в JSON. Генератор нижче бере справжні замовлення дня і «псує» їх так, як буває в житті: повтори (застосунок відправив двічі), `"412,50"` з комою, `n/a` замість суми, від'ємні суми, порожній або неіснуючий ресторан, ` Delivered` з пробілом і великою літерою. Множина `BROKEN` запам'ятовує, які `id` зіпсовано безповоротно, — з нею звіримо результат очищення.

In [ ]:
BRONZE = LAKE / "bronze"
BROKEN = {"orders": set(), "rides": set()}       # «правда» генератора для перевірок
ORDER_FIELDS = ["order_id", "ts", "restaurant_id", "driver_id", "customer_id", "items", "total", "status"]
RIDE_FIELDS = ["ride_id", "ts", "driver_id", "from_district", "to_district", "distance_km", "fare"]


def add_duplicates(raw, rng, share=0.02):
    """Застосунок інколи надсилає той самий запис двічі: додаємо копії й перемішуємо."""
    copies = raw.iloc[rng.choice(len(raw), size=int(len(raw) * share), replace=False)]
    both = pd.concat([raw, copies])
    return both.iloc[rng.permutation(len(both))]


def dirty_orders_day(day_orders, rng):
    raw = day_orders[ORDER_FIELDS].astype(str)
    n = len(raw)
    comma = rng.random(n) < 0.02                       # "412,50" — можна виправити
    raw.loc[comma, "total"] = raw.loc[comma, "total"].str.replace(".", ",", regex=False)
    odd_status = rng.random(n) < 0.03                  # " Delivered" — можна виправити
    raw.loc[odd_status, "status"] = " " + raw.loc[odd_status, "status"].str.capitalize()
    bad_total = rng.random(n) < 0.005                  # "n/a" — не виправити
    raw.loc[bad_total, "total"] = "n/a"
    negative = (rng.random(n) < 0.003) & ~bad_total    # "-412.5" — не виправити
    raw.loc[negative, "total"] = "-" + raw.loc[negative, "total"]
    no_restaurant = rng.random(n) < 0.005              # порожньо — не виправити
    raw.loc[no_restaurant, "restaurant_id"] = ""
    unknown = (rng.random(n) < 0.002) & ~no_restaurant # ресторану 999 немає
    raw.loc[unknown, "restaurant_id"] = "999"
    BROKEN["orders"].update(day_orders.loc[bad_total | negative | no_restaurant | unknown, "order_id"].tolist())
    return add_duplicates(raw, rng)


def dirty_rides_day(day_rides, rng):
    raw = day_rides[RIDE_FIELDS].astype({"ts": str}).astype(object)
    n = len(raw)
    comma = rng.random(n) < 0.02                       # "3,4" замість 3.4 — можна виправити
    raw.loc[comma, "distance_km"] = [str(v).replace(".", ",") for v in raw.loc[comma, "distance_km"]]
    no_fare = rng.random(n) < 0.005                    # null — не виправити
    raw.loc[no_fare, "fare"] = None
    bad_km = (rng.random(n) < 0.003) & ~comma          # 0 км — не виправити
    raw.loc[bad_km, "distance_km"] = 0
    unknown = rng.random(n) < 0.003                    # водія 999 немає
    raw.loc[unknown, "driver_id"] = 999
    BROKEN["rides"].update(day_rides.loc[no_fare | bad_km | unknown, "ride_id"].tolist())
    return add_duplicates(raw, rng)


def write_bronze_day(day, rng, root=BRONZE):
    """Сирі файли одного дня: orders_<день>.csv і rides_<день>.json."""
    (root / "orders").mkdir(parents=True, exist_ok=True)
    (root / "rides").mkdir(parents=True, exist_ok=True)
    dirty_orders_day(orders[orders["order_date"] == day], rng).to_csv(root / "orders" / f"orders_{day}.csv", index=False)
    dirty_rides_day(rides[rides["ride_date"] == day], rng).to_json(
        root / "rides" / f"rides_{day}.json", orient="records", lines=True, force_ascii=False)


ALL_DAYS = sorted(orders["order_date"].unique())
FIRST_WEEKS, LAST_WEEK = ALL_DAYS[:-7], ALL_DAYS[-7:]  # останній тиждень «прийде» пізніше (приклад 7.8)

In [ ]:
bronze_rng = np.random.default_rng(2026)
for day in FIRST_WEEKS:
    write_bronze_day(day, bronze_rng)
tree(BRONZE, limit=4)
print()
with open(BRONZE / "orders" / "orders_2026-09-01.csv", encoding="utf-8") as f:
    for _ in range(6):
        print(f.readline().rstrip())

У bronze — 54 дні: по CSV і JSON на день. Рядки перемішані, серед них — повтори й сміття. Ми **нічого тут не змінюємо**: bronze — як касова стрічка, її не переписують.

### Приклад 7.2. Schema-on-read: профіль сирих даних

Перш ніж чистити, з'ясуємо, **що саме** зламано. Читаємо всі CSV як текст (`all_varchar = true`), щоб жоден рядок не загубився через помилку типу, і рахуємо проблеми через `count(*) FILTER (WHERE …)` — підрахунок лише рядків, що задовольняють умову.

In [ ]:
PROFILE_SQL = f"""
SELECT count(*)                                                   AS rows,
       count(DISTINCT order_id)                                   AS unique_ids,
       count(*) FILTER (WHERE total LIKE '%,%')                   AS comma_total,
       count(*) FILTER (WHERE TRY_CAST(replace(total, ',', '.') AS DECIMAL(10, 2)) IS NULL) AS bad_total,
       count(*) FILTER (WHERE total LIKE '-%')                    AS negative,
       count(*) FILTER (WHERE restaurant_id IS NULL)              AS no_restaurant,
       count(*) FILTER (WHERE status NOT IN ('delivered', 'cancelled', 'returned')) AS odd_status
FROM read_csv('{BRONZE}/orders/*.csv', all_varchar = true)
"""

In [ ]:
con.sql(PROFILE_SQL)

Тепер відомо, які правила потрібні silver-шару: прибрати повтори (`rows` > `unique_ids`), виправити кому й регістр статусу, а рядки з неможливою сумою чи без ресторану — **відкласти в карантин** з причиною, а не мовчки викинути.

### Приклад 7.3. Пакет конвеєра: чисте перетворення

Код конвеєра — не в ноутбуці, а в **пакеті** (модулі — урок 13, пакет `dispatch` — урок 29), розкладеному за ролями:

```text
duckdb_lab/
├── smachno_lake/
│   ├── __init__.py
│   ├── transform.py   ← чисті функції: DataFrame → DataFrame (без файлів) — легко тестувати
│   ├── pipeline.py    ← extract і load: читає bronze, пише silver і quarantine
│   └── quality.py     ← перевірки якості даних
└── tests/
    └── test_transform.py
```

`transform.py` нічого не знає про файли: на вході сирий DataFrame, на виході — пара «чисті, відкинуті». Усередині — SQL у **власній** DuckDB-базі в пам'яті, щоб функція не залежала від зовнішнього стану.

In [ ]:
PACKAGE = LAB / "smachno_lake"
(PACKAGE).mkdir(exist_ok=True)
(LAB / "tests").mkdir(exist_ok=True)
(PACKAGE / "__init__.py").write_text('"""Конвеєр озера даних «Смачно + Таксі»."""\n', encoding="utf-8")
if str(LAB) not in sys.path:
    sys.path.insert(0, str(LAB))         # щоб import smachno_lake знайшов пакет у duckdb_lab/
print("папки уроку:", sorted(p.name for p in LAB.iterdir() if p.is_dir()))

In [ ]:
%%writefile duckdb_lab/smachno_lake/transform.py
"""Silver-шар: чисті перетворення «сирий DataFrame → (чисті, відкинуті)».

Без файлів, мережі й глобального стану — тому функції легко тестувати.
"""
import duckdb
import pandas as pd

VALID_STATUSES = ("delivered", "cancelled", "returned")

TYPED_ORDERS_SQL = """
SELECT
    TRY_CAST(order_id AS BIGINT)       AS order_id,
    TRY_CAST(ts AS TIMESTAMP)          AS ts,
    TRY_CAST(restaurant_id AS INTEGER) AS restaurant_id,
    TRY_CAST(driver_id AS INTEGER)     AS driver_id,
    TRY_CAST(customer_id AS INTEGER)   AS customer_id,
    TRY_CAST(items AS INTEGER)         AS items,
    TRY_CAST(replace(replace(trim(total), ' ', ''), ',', '.') AS DECIMAL(10, 2)) AS total,
    lower(trim(status))                AS status,
    total                              AS total_raw
FROM raw
QUALIFY row_number() OVER (PARTITION BY order_id ORDER BY ts) = 1   -- повтори: лишаємо один запис
"""

CHECKED_SQL = """
SELECT *,
    CASE
        WHEN order_id IS NULL OR ts IS NULL                         THEN 'bad_id_or_time'
        WHEN restaurant_id IS NULL                                  THEN 'no_restaurant'
        WHEN restaurant_id NOT IN (SELECT restaurant_id FROM known) THEN 'unknown_restaurant'
        WHEN total IS NULL                                          THEN 'bad_total'
        WHEN total <= 0                                             THEN 'non_positive_total'
        WHEN NOT list_contains(?, status)                           THEN 'bad_status'
    END AS reject_reason
FROM typed
"""


def clean_orders(raw: pd.DataFrame, known_restaurants) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Сирі замовлення (усе текстом) → (чисті рядки, відкинуті рядки з причиною)."""
    con = duckdb.connect()                          # своя порожня база в пам'яті
    con.register("raw", raw)
    con.register("known", pd.DataFrame({"restaurant_id": list(known_restaurants)}))
    con.execute(f"CREATE TEMP TABLE typed AS {TYPED_ORDERS_SQL}")
    con.execute(f"CREATE TEMP TABLE checked AS {CHECKED_SQL}", [list(VALID_STATUSES)])
    good = con.sql("SELECT * EXCLUDE (total_raw, reject_reason) FROM checked "
                   "WHERE reject_reason IS NULL ORDER BY order_id").df()
    rejected = con.sql("SELECT order_id, restaurant_id, total_raw, status, reject_reason FROM checked "
                       "WHERE reject_reason IS NOT NULL ORDER BY order_id").df()
    con.close()
    return good, rejected

Два кроки SQL: `TYPED_ORDERS_SQL` перетворює текст на типи (`TRY_CAST` — без падінь) і прибирає повтори через `QUALIFY row_number() … = 1` з розділу 3; `CHECKED_SQL` для кожного рядка пише **першу** причину відмови або `NULL`. Список дозволених статусів передано параметром `?`.

### Приклад 7.4. Чисте перетворення на одному дні

In [ ]:
from smachno_lake.transform import clean_orders

KNOWN_RESTAURANTS = restaurants["restaurant_id"].tolist()
raw_day = con.sql(f"SELECT * FROM read_csv('{BRONZE}/orders/orders_2026-09-01.csv', all_varchar = true)").df()

In [ ]:
good_day, rejected_day = clean_orders(raw_day, KNOWN_RESTAURANTS)
print(f"сирих рядків: {len(raw_day)}, чистих: {len(good_day)}, відкинуто: {len(rejected_day)}")
print(rejected_day["reject_reason"].value_counts().to_string())
print(good_day.dtypes.to_string())

З кожного сирого рядка вийшов або чистий, або відкинутий з причиною; повтори зникли. `total` у pandas — `float64` (DuckDB перетворює `DECIMAL` для pandas), тож під час запису в silver повернемо точний тип `DECIMAL(10, 2)`.

### Приклад 7.5. Тест для перетворення

Чисту функцію перевіряємо **маленьким** DataFrame, де відомий кожен рядок, — як у уроці 26 з pytest. Тест живе у файлі, запускаємо його командою `python -m pytest`.

In [ ]:
%%writefile duckdb_lab/tests/test_transform.py
import pandas as pd

from smachno_lake.transform import clean_orders

RAW = pd.DataFrame({
    "order_id":      ["1", "1", "2", "3", "4", "5"],
    "ts":            ["2026-09-01 10:00:00"] * 6,
    "restaurant_id": ["7", "7", "7", None, "999", "7"],
    "driver_id":     ["3"] * 6,
    "customer_id":   ["100"] * 6,
    "items":         ["2"] * 6,
    "total":         ["412,50", "412,50", "n/a", "300", "250", "-10"],
    "status":        [" Delivered", " Delivered", "delivered", "delivered", "cancelled", "delivered"],
})


def test_duplicates_removed_and_values_fixed():
    good, _ = clean_orders(RAW, known_restaurants=[7])
    assert good["order_id"].tolist() == [1]
    assert good.loc[0, "total"] == 412.50
    assert good.loc[0, "status"] == "delivered"


def test_every_bad_row_has_reason():
    _, rejected = clean_orders(RAW, known_restaurants=[7])
    assert dict(zip(rejected["order_id"], rejected["reject_reason"])) == {
        2: "bad_total", 3: "no_restaurant", 4: "unknown_restaurant", 5: "non_positive_total"}


def test_nothing_lost():
    good, rejected = clean_orders(RAW, known_restaurants=[7])
    assert len(good) + len(rejected) == RAW["order_id"].nunique()

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "tests"],
                        cwd=LAB, capture_output=True, text=True)
print(result.stdout.strip().splitlines()[-1])
assert result.returncode == 0, result.stdout

Три тести: повтори прибрано й значення виправлено; кожен поганий рядок має причину; жоден рядок не загубився (`чисті + відкинуті = унікальні`). Останній — головний інваріант будь-якого очищення.

### Приклад 7.6. Extract і load: ідемпотентне завантаження

Конвеєр перезапускають: впав посередині, виправили правило, дозавантажили день. **Ідемпотентне** завантаження — таке, що повторний запуск для того самого дня дає той самий результат, а не подвоєні дані. Найпростіший спосіб у файловому озері — **перезаписати розділ дня цілком**: новий файл пишемо в тимчасову папку поруч, потім старий розділ видаляємо, а тимчасову перейменовуємо.

In [ ]:
%%writefile duckdb_lab/smachno_lake/pipeline.py
"""Extract і load: усе, що торкається файлів. Логіка очищення — у transform.py."""
import shutil
from datetime import date
from pathlib import Path

import duckdb
import pandas as pd

from .transform import clean_orders

SILVER_SELECT = """
SELECT order_id::BIGINT AS order_id, ts::TIMESTAMP AS ts, restaurant_id::INTEGER AS restaurant_id,
       driver_id::INTEGER AS driver_id, customer_id::INTEGER AS customer_id, items::INTEGER AS items,
       total::DECIMAL(10, 2) AS total, status::VARCHAR AS status
FROM data
"""


def bronze_days(lake: Path) -> list[date]:
    files = (lake / "bronze" / "orders").glob("orders_*.csv")
    return sorted(date.fromisoformat(p.stem.removeprefix("orders_")) for p in files)


def silver_days(lake: Path) -> list[date]:
    folders = (lake / "silver" / "orders").glob("order_date=*")
    return sorted(date.fromisoformat(p.name.split("=", 1)[1]) for p in folders if (p / "data.parquet").exists())


def pending_days(lake: Path) -> list[date]:
    """Інкрементне завантаження: дні, що є в bronze, але ще не потрапили в silver."""
    done = set(silver_days(lake))
    return [day for day in bronze_days(lake) if day not in done]


def extract_orders(lake: Path, day: date) -> pd.DataFrame:
    path = lake / "bronze" / "orders" / f"orders_{day}.csv"
    with duckdb.connect() as con:
        return con.sql(f"SELECT * FROM read_csv('{path}', all_varchar = true)").df()


def write_partition(data: pd.DataFrame, target: Path, select_sql: str = "SELECT * FROM data") -> Path:
    """Ідемпотентно записати розділ: спершу в target.tmp, потім замінити target цілком."""
    tmp = target.with_name(target.name + ".tmp")
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True)
    with duckdb.connect() as con:
        con.register("data", data)
        con.execute(f"COPY ({select_sql}) TO '{tmp / 'data.parquet'}' (FORMAT PARQUET)")
    shutil.rmtree(target, ignore_errors=True)
    tmp.rename(target)
    return target


def load_day(lake: Path, day: date, known_restaurants) -> dict:
    raw = extract_orders(lake, day)
    good, rejected = clean_orders(raw, known_restaurants)
    write_partition(good, lake / "silver" / "orders" / f"order_date={day}", SILVER_SELECT)
    write_partition(rejected, lake / "quarantine" / "orders" / f"order_date={day}")
    return {"day": day, "raw": len(raw), "silver": len(good), "rejected": len(rejected)}


def run_pending(lake: Path, known_restaurants) -> list[dict]:
    return [load_day(lake, day, known_restaurants) for day in pending_days(lake)]

Для порівняння — **наївне** завантаження, яке щоразу дописує новий файл у розділ дня. Запустимо обидва варіанти двічі для 1 вересня.

In [ ]:
import uuid

from smachno_lake import pipeline


def load_day_append(lake, day, known_restaurants):
    """❌ Не ідемпотентне: кожен запуск додає ще один файл з тими самими рядками."""
    good, _ = clean_orders(pipeline.extract_orders(lake, day), known_restaurants)
    folder = lake / "silver_append" / "orders" / f"order_date={day}"
    folder.mkdir(parents=True, exist_ok=True)
    with duckdb.connect() as tmp_con:
        tmp_con.register("data", good)
        tmp_con.execute(f"COPY ({pipeline.SILVER_SELECT}) TO '{folder / f'part-{uuid.uuid4().hex[:8]}.parquet'}' (FORMAT PARQUET)")


def count_rows(folder):
    return con.sql(f"SELECT count(*) FROM read_parquet('{folder}/*/*.parquet', hive_partitioning = true)").fetchone()[0]

In [ ]:
day = date(2026, 9, 1)
for run in (1, 2):
    load_day_append(LAKE, day, KNOWN_RESTAURANTS)
    print(f"наївне, запуск {run}:       {count_rows(LAKE / 'silver_append' / 'orders')} рядків")
for run in (1, 2):
    pipeline.load_day(LAKE, day, KNOWN_RESTAURANTS)
    print(f"ідемпотентне, запуск {run}: {count_rows(LAKE / 'silver' / 'orders')} рядків")

Наївне завантаження подвоїло день — і подвоїло б виторг у кожному звіті. Ідемпотентне дало ті самі рядки. Запис через `.tmp` захищає й від падіння посередині: поки новий файл не дописано, старий розділ лишається цілим.

Інший поширений спосіб ідемпотентності — у таблиці бази: `DELETE` рядків дня і `INSERT` нових **в одній транзакції** (приклад 8.4).

### Приклад 7.7. Перший запуск конвеєра

`run_pending` завантажує всі дні, яких ще немає в silver. 1 вересня вже є — його пропустить.

In [ ]:
start = time.perf_counter()
report = pd.DataFrame(pipeline.run_pending(LAKE, KNOWN_RESTAURANTS))
print(f"завантажено днів: {len(report)} за {time.perf_counter() - start:.1f} с; перший — {report['day'].iloc[0]}")
print(report[["raw", "silver", "rejected"]].sum().to_string())
print("днів у silver:", len(pipeline.silver_days(LAKE)))

### Приклад 7.8. Інкрементне завантаження

Минув тиждень: у bronze з'явилися файли за 25–31 жовтня. **Інкрементне** завантаження обробляє лише нове, а не перераховує два місяці. `pending_days` порівнює дні в bronze і в silver.

In [ ]:
for day in LAST_WEEK:
    write_bronze_day(day, bronze_rng)
print("чекають обробки:", [str(d) for d in pipeline.pending_days(LAKE)])

In [ ]:
new = pipeline.run_pending(LAKE, KNOWN_RESTAURANTS)
print("оброблено днів:", len(new), "| у silver тепер:", len(pipeline.silver_days(LAKE)))
print("повторний запуск:", pipeline.run_pending(LAKE, KNOWN_RESTAURANTS), "— нічого нового")

Інкремент «за днями» працює, бо кожен день — окремий розділ. Якщо дані дня можуть **запізнюватися** (поїздка вчорашнього вечора прийшла сьогодні), перезавантажують кілька останніх днів — завдяки ідемпотентності це безпечно.

### Приклад 7.9. Перевірки якості як код

Перевірка якості — функція, що рахує порушення правила і повертає опис проблеми або `None`. Набір таких функцій запускають після кожного завантаження: краще впасти з повідомленням, ніж мовчки показати власниці неправильний виторг. Так працюють бібліотеки `dbt tests`, Great Expectations, Soda — ми зробимо найпростішу версію.

In [ ]:
%%writefile duckdb_lab/smachno_lake/quality.py
"""Перевірки якості даних. Кожна повертає None (усе гаразд) або текст проблеми.

Імена таблиць і стовпців підставляються f-рядком — лише з коду конвеєра, не від користувача.
"""


def check_not_null(con, table, column):
    bad = con.sql(f"SELECT count(*) FROM {table} WHERE {column} IS NULL").fetchone()[0]
    return f"{table}.{column}: {bad} порожніх" if bad else None


def check_unique(con, table, column):
    bad = con.sql(f"SELECT count({column}) - count(DISTINCT {column}) FROM {table}").fetchone()[0]
    return f"{table}.{column}: {bad} повторів" if bad else None


def check_range(con, table, column, low, high):
    bad = con.execute(f"SELECT count(*) FROM {table} WHERE {column} NOT BETWEEN ? AND ?", [low, high]).fetchone()[0]
    return f"{table}.{column}: {bad} значень поза [{low}, {high}]" if bad else None


def check_accepted(con, table, column, values):
    bad = con.execute(f"SELECT count(*) FROM {table} WHERE NOT list_contains(?, {column})", [list(values)]).fetchone()[0]
    return f"{table}.{column}: {bad} недозволених значень" if bad else None


def check_foreign_key(con, table, column, ref_table, ref_column):
    bad = con.sql(f"SELECT count(*) FROM {table} WHERE {column} NOT IN (SELECT {ref_column} FROM {ref_table})").fetchone()[0]
    return f"{table}.{column}: {bad} посилань у нікуди ({ref_table})" if bad else None


def run_checks(con, checks):
    """checks — список кортежів (функція, аргументи…). Повертає список проблем: порожній — дані в порядку."""
    return [problem for check, *args in checks if (problem := check(con, *args))]

In [ ]:
from smachno_lake.quality import (check_accepted, check_foreign_key, check_not_null, check_range,
                                  check_unique, run_checks)

SILVER_ORDERS = LAKE / "silver" / "orders"
SILVER_VIEW = f"CREATE OR REPLACE VIEW silver_orders AS SELECT * FROM read_parquet('{SILVER_ORDERS}/*/*.parquet', hive_partitioning = true)"
BRONZE_VIEW = f"CREATE OR REPLACE VIEW bronze_orders AS SELECT * FROM read_csv('{BRONZE}/orders/*.csv', all_varchar = true)"

def order_checks(table):
    """Однакові правила для будь-якої таблиці замовлень — сирої чи чистої."""
    return [
        (check_not_null, table, "order_id"),
        (check_unique, table, "order_id"),
        (check_not_null, table, "restaurant_id"),
        (check_accepted, table, "status", ["delivered", "cancelled", "returned"]),
    ]


SILVER_ONLY = [
    (check_range, "silver_orders", "total", 0.01, 50_000),
    (check_foreign_key, "silver_orders", "restaurant_id", "restaurants", "restaurant_id"),
]

In [ ]:
con.sql(SILVER_VIEW)
con.sql(BRONZE_VIEW)
print("silver:", run_checks(con, order_checks("silver_orders") + SILVER_ONLY) or "усі перевірки пройдено ✅")
print("bronze:")
for problem in run_checks(con, order_checks("bronze_orders")):
    print("  ⚠️", problem)

Ті самі правила на bronze знаходять три проблеми, на silver — жодної: очищення спрацювало. У справжньому конвеєрі `run_checks` після завантаження кидає виняток, і дані з помилками не доходять до gold.

### Приклад 7.10. Звірка з «правдою»

Bronze ми зіпсували самі, тож знаємо відповідь: у silver мають бути **всі** замовлення, крім зіпсованих безповоротно, без жодного повтору, і з тими самими сумами.

In [ ]:
silver_ids = set(con.sql("SELECT order_id FROM silver_orders").df()["order_id"])
expected_ids = set(orders["order_id"]) - BROKEN["orders"]
assert silver_ids == expected_ids, "silver має містити рівно незіпсовані замовлення"

silver_sum = con.sql("SELECT sum(total) FROM silver_orders").fetchone()[0]
expected_sum = orders.loc[orders["order_id"].isin(expected_ids), "total"].sum()
assert abs(float(silver_sum) - expected_sum) < 0.01
quarantined = con.sql(f"SELECT count(*) FROM read_parquet('{LAKE}/quarantine/orders/*/*.parquet')").fetchone()[0]
assert quarantined == len(BROKEN["orders"]), "у карантині — рівно зіпсовані замовлення"
print(f"silver: {len(silver_ids)} замовлень, карантин: {quarantined}, зіпсовано генератором: {len(BROKEN['orders'])}")
print(f"сума silver = сума оригіналу: {float(silver_sum):,.2f} грн".replace(",", " "))

Silver збігся з оригіналом до копійки: повтори прибрано, коми й статуси виправлено, а в карантин потрапили рівно ті рядки, які виправити неможливо.

### Задача 2. Silver для поїздок таксі

Сирі поїздки лежать у `bronze/rides/rides_<день>.json`. Клітинка нижче читає один день як текст. Допиши `clean_rides(raw, known_drivers)` за зразком `clean_orders`:

- повтори за `ride_id` — лишити один запис;
- `distance_km` буває з комою (`"3,4"`) — виправити й перетворити на `DOUBLE`; `fare` — на `DOUBLE`; `ride_id` — `BIGINT`, `driver_id` — `INTEGER`, `ts` — `TIMESTAMP`;
- відкинути з причиною `reject_reason`: `'no_fare'` (немає `fare`), `'bad_distance'` (немає відстані або `distance_km <= 0`), `'unknown_driver'` (`driver_id` не з `known_drivers`);
- повернути пару `(good, rejected)`: `good` — стовпці `ride_id, ts, driver_id, from_district, to_district, distance_km, fare`; `rejected` — щонайменше `ride_id, reject_reason`.

Перевірка звіряє результат з «правдою» генератора і запускає перевірки якості з прикладу 7.9.

In [ ]:
raw_rides = con.sql(f"""
    SELECT * FROM read_json('{BRONZE}/rides/rides_2026-10-03.json',
        columns = {{ride_id: 'VARCHAR', ts: 'VARCHAR', driver_id: 'VARCHAR', from_district: 'VARCHAR',
                   to_district: 'VARCHAR', distance_km: 'VARCHAR', fare: 'VARCHAR'}})
""").df()
KNOWN_DRIVERS = drivers["driver_id"].tolist()
print(len(raw_rides), "сирих поїздок;", raw_rides["distance_km"].str.contains(",").sum(), "з комою")

In [ ]:
def clean_rides(raw, known_drivers):
    # YOUR CODE HERE
    pass


In [ ]:
cleaned = clean_rides(raw_rides, KNOWN_DRIVERS)
assert isinstance(cleaned, tuple) and len(cleaned) == 2, "clean_rides має повертати пару (good, rejected)"
good_rides, rejected_rides = cleaned
assert list(good_rides.columns) == ["ride_id", "ts", "driver_id", "from_district", "to_district", "distance_km", "fare"], \
    f"стовпці good: {list(good_rides.columns)}"
assert {"ride_id", "reject_reason"} <= set(rejected_rides.columns), "у rejected потрібні ride_id і reject_reason"

day_ids = set(rides.loc[rides["ride_date"] == date(2026, 10, 3), "ride_id"])
assert set(good_rides["ride_id"]) == day_ids - BROKEN["rides"], "у good мають бути рівно незіпсовані поїздки дня"
assert set(rejected_rides["ride_id"]) == day_ids & BROKEN["rides"], "у rejected — рівно зіпсовані поїздки"
assert set(rejected_rides["reject_reason"]) <= {"no_fare", "bad_distance", "unknown_driver"}
assert good_rides["distance_km"].dtype.kind == "f" and good_rides["fare"].dtype.kind == "f", "distance_km і fare — числа"

con.register("good_rides", good_rides)
problems = run_checks(con, [(check_unique, "good_rides", "ride_id"), (check_range, "good_rides", "distance_km", 0.1, 200),
                            (check_foreign_key, "good_rides", "driver_id", "drivers", "driver_id")])
assert problems == [], problems
print(f"чистих: {len(good_rides)}, відкинуто: {len(rejected_rides)}", rejected_rides["reject_reason"].value_counts().to_dict())
print("✅ Задача 2 розв'язана")

---
## 8. Gold і зіркова схема

Silver — чисті **події**. Але власниці потрібні відповіді: «виторг по кухнях у вихідні», «найкращий кур'єр місяця». Для таких запитів дані організовують за **зірковою схемою** (star schema, Ральф Кімбол):

```text
                 dim_date                         dim_restaurant
          (date_key, weekday, is_weekend)     (restaurant_id, name, cuisine, district)
                        \                          /
                         \                        /
                          ───── fact_orders ─────
                          (order_id, date_key, restaurant_id, driver_id,
                           items, total, status)        ← одна подія = один рядок
                         /
                        /
                 dim_driver
          (driver_id, name, home_district, vehicle)
```

- **таблиця фактів** (`fact_`) — події з числами, які сумують: кількість, сума. Довга й вузька;
- **таблиці вимірів** (`dim_`) — «хто, де, коли»: описи, за якими групують і фільтрують. Короткі й широкі.

Питання до зірки завжди однакове за формою: факти `JOIN` потрібні виміри → `WHERE` → `GROUP BY` атрибути вимірів → агрегати фактів.

### Приклад 8.1. Сховище: виміри й факти

Окремий файл `warehouse.duckdb` — аналітичне сховище. Виміри — таблиці (вони маленькі), факти — **подання** прямо над Parquet-файлами silver: дані не копіюються, сховище читає озеро. Це і є ідея lakehouse.

In [ ]:
WAREHOUSE = LAB / "warehouse.duckdb"
STAR_SQL = [
    "CREATE OR REPLACE TABLE dim_restaurant AS SELECT restaurant_id, name, cuisine, district FROM restaurants",
    "CREATE OR REPLACE TABLE dim_driver AS SELECT driver_id, name, district AS home_district, vehicle FROM drivers",
    """CREATE OR REPLACE TABLE dim_date AS
       SELECT d::DATE AS date_key, year(d) AS year, month(d) AS month, isodow(d) AS weekday_num,
              ['пн', 'вт', 'ср', 'чт', 'пт', 'сб', 'нд'][isodow(d)] AS weekday, isodow(d) >= 6 AS is_weekend
       FROM range(DATE '2026-09-01', DATE '2026-11-01', INTERVAL 1 DAY) AS t(d)""",
    f"""CREATE OR REPLACE VIEW fact_orders AS
        SELECT order_id, order_date AS date_key, restaurant_id, driver_id, customer_id, items, total, status
        FROM read_parquet('{SILVER_ORDERS}/*/*.parquet', hive_partitioning = true)""",
]


def build_warehouse(path):
    wh = duckdb.connect(str(path))
    for statement in STAR_SQL:
        wh.sql(statement)
    return wh

In [ ]:
wh = build_warehouse(WAREHOUSE)
wh.sql("SELECT table_name, table_type FROM information_schema.tables ORDER BY table_name")

### Приклад 8.2. Запит до зірки

«Скільки доставлених замовлень на день і який середній чек — для кожного дня тижня?»

In [ ]:
WEEKDAY_SQL = """
SELECT d.weekday_num, d.weekday,
       round(count(*) / count(DISTINCT f.date_key))::INTEGER AS orders_per_day,
       round(avg(f.total), 2)                        AS avg_check
FROM fact_orders AS f
JOIN dim_date AS d USING (date_key)
WHERE f.status = 'delivered'
GROUP BY ALL
ORDER BY d.weekday_num
"""

In [ ]:
wh.sql(WEEKDAY_SQL)

У суботу й неділю замовлень на третину більше, а середній чек той самий — як і закладено в генератор. Питання «а по кухнях?» — ще один `JOIN dim_restaurant` і стовпець у `GROUP BY`; схема не змінюється.

### Приклад 8.3. Gold-вітрини

**Вітрина** (data mart) — готова маленька таблиця під конкретний звіт. Дашборд читає її миттєво, не сканує сотні тисяч фактів. Запишемо дві вітрини в `gold/`.

In [ ]:
GOLD = LAKE / "gold"
MARTS = {
    "daily_district": """
        SELECT f.date_key, r.district, count(*) AS orders, round(sum(f.total), 2) AS revenue,
               round(avg(f.total), 2) AS avg_check
        FROM fact_orders AS f JOIN dim_restaurant AS r USING (restaurant_id)
        WHERE f.status = 'delivered'
        GROUP BY ALL ORDER BY ALL""",
    "driver_monthly": """
        SELECT d.year, d.month, dr.name AS driver, dr.vehicle, count(*) AS orders, round(sum(f.total), 2) AS revenue
        FROM fact_orders AS f JOIN dim_driver AS dr USING (driver_id) JOIN dim_date AS d USING (date_key)
        WHERE f.status = 'delivered'
        GROUP BY ALL ORDER BY ALL""",
}


def publish_marts(wh):
    GOLD.mkdir(exist_ok=True)
    for name, sql in MARTS.items():
        wh.sql(f"COPY ({sql}) TO '{GOLD / f'{name}.parquet'}' (FORMAT PARQUET)")

In [ ]:
publish_marts(wh)
tree(GOLD)
print(wh.sql(f"""
    SELECT district, round(sum(revenue) / 1e6, 2) AS revenue_mln, sum(orders)::BIGINT AS orders
    FROM '{GOLD / "daily_district.parquet"}'
    GROUP BY ALL ORDER BY revenue_mln DESC
"""))

Вітрина `daily_district` — 366 рядків замість 330 тисяч, і звіт по районах з неї — те саме, що в прикладі 1.3, але без `JOIN` і сканування фактів.

### Приклад 8.4. Ідемпотентна публікація в таблицю: DELETE + INSERT в одній транзакції

Якщо вітрина живе **таблицею** в базі, ідемпотентність дає транзакція (урок 30, розділ «Транзакції»): видалити рядки дня і вставити нові — разом або ніяк. Повторний запуск дає той самий результат.

In [ ]:
MART_TABLE = """
CREATE TABLE IF NOT EXISTS mart_daily_district (
    date_key DATE, district VARCHAR, orders BIGINT, revenue DECIMAL(14, 2), avg_check DECIMAL(10, 2),
    PRIMARY KEY (date_key, district)
)
"""
INSERT_DAY = """
INSERT INTO mart_daily_district
SELECT f.date_key, r.district, count(*), sum(f.total), avg(f.total)
FROM fact_orders AS f JOIN dim_restaurant AS r USING (restaurant_id)
WHERE f.status = 'delivered' AND f.date_key = ?
GROUP BY ALL
"""


def publish_day(wh, day):
    """Перерахувати вітрину за один день: DELETE + INSERT атомарно."""
    wh.execute("BEGIN TRANSACTION")
    try:
        wh.execute("DELETE FROM mart_daily_district WHERE date_key = ?", [day])
        wh.execute(INSERT_DAY, [day])
        wh.execute("COMMIT")
    except Exception:
        wh.execute("ROLLBACK")
        raise
    return wh.execute("SELECT count(*) FROM mart_daily_district").fetchone()[0]

In [ ]:
wh.sql(MART_TABLE)
for run in (1, 2):
    print(f"запуск {run}: рядків у вітрині — {publish_day(wh, date(2026, 10, 3))}")
for day in ALL_DAYS:
    publish_day(wh, day)
print("після всіх 61 днів:", wh.sql("SELECT count(*) FROM mart_daily_district").fetchone()[0])

Без `DELETE` другий запуск упав би на `PRIMARY KEY (date_key, district)` — і це теж захист: первинний ключ вітрини не дає задублювати день.

### Приклад 8.5. Каталог даних

Коли в озері десятки наборів даних, потрібна відповідь на «що тут є, звідки воно, яка схема, коли оновлено?». Це **каталог** (metadata catalog). Найпростіший — файл `catalog.json`, який конвеєр оновлює після кожного запуску. Великі системи — AWS Glue Data Catalog, Unity Catalog, каталоги Iceberg — роблять те саме в масштабі компанії.

In [ ]:
DATASETS = [
    ("orders", "bronze", f"{BRONZE}/orders/*.csv", "csv", "—"),
    ("orders", "silver", f"{SILVER_ORDERS}/*/*.parquet", "parquet", "order_date"),
    ("orders_rejected", "quarantine", f"{LAKE}/quarantine/orders/*/*.parquet", "parquet", "order_date"),
    ("daily_district", "gold", f"{GOLD}/daily_district.parquet", "parquet", "—"),
    ("driver_monthly", "gold", f"{GOLD}/driver_monthly.parquet", "parquet", "—"),
]


def describe_dataset(con, name, layer, pattern, fmt, partition):
    reader = "read_csv(?, all_varchar = true)" if fmt == "csv" else "read_parquet(?, hive_partitioning = true)"
    relation = con.sql(f"SELECT * FROM {reader}", params=[pattern])
    files = sorted(Path().glob(pattern))
    return {
        "dataset": name, "layer": layer, "path": pattern, "format": fmt, "partitioned_by": partition,
        "files": len(files), "rows": con.execute(f"SELECT count(*) FROM {reader}", [pattern]).fetchone()[0],
        "size_mb": round(sum(f.stat().st_size for f in files) / 1e6, 2),
        "columns": dict(zip(relation.columns, map(str, relation.types))),
        "updated_at": time.strftime("%Y-%m-%d %H:%M", time.localtime(max(f.stat().st_mtime for f in files))),
    }


def build_catalog(con, path):
    catalog = [describe_dataset(con, *dataset) for dataset in DATASETS]
    path.write_text(json.dumps(catalog, ensure_ascii=False, indent=2), encoding="utf-8")
    return catalog

In [ ]:
catalog = build_catalog(con, LAKE / "catalog.json")
print(pd.DataFrame(catalog).drop(columns=["path", "columns", "updated_at"]).to_string(index=False))
print("схема silver.orders:", catalog[1]["columns"])

Каталог одразу показує історію даних: bronze — 61 CSV з усім текстом і повторами; silver — 61 розділ Parquet з типами; карантин; дві маленькі gold-вітрини. Шлях до файлу тут передано параметром `?` — функції `read_csv` / `read_parquet` приймають його так само, як значення.

### Задача 3. Ідемпотентне завантаження поїздок за день

Додай поїздки таксі в silver. Допиши `load_rides_day(rides, day, lake)`:

- бере з DataFrame `rides` (розділ 0) поїздки лише за день `day`;
- записує їх у `lake / "silver" / "rides" / f"ride_date={day}" / "data.parquet"`;
- стовпця `ride_date` **у файлі немає** — він у імені папки (Hive-розбиття);
- повторний запуск для того самого дня **не дублює** рядки і **не чіпає** інші дні (використай `pipeline.write_partition` з прикладу 7.6 або напиши запис сам);
- повертає кількість записаних рядків.

Підказка: `pipeline.write_partition(data, target, select_sql)` реєструє DataFrame під іменем `data`; `SELECT * EXCLUDE (ride_date) FROM data`.

In [ ]:
def load_rides_day(rides, day, lake):
    # YOUR CODE HERE
    pass


In [ ]:
RIDES_SILVER = LAKE / "silver" / "rides"
first_day, second_day = date(2026, 10, 3), date(2026, 10, 4)
expected = int((rides["ride_date"] == first_day).sum())

written = [load_rides_day(rides, first_day, LAKE) for _ in range(2)]
assert written == [expected, expected], f"має повертати кількість рядків дня ({expected}), а повернула {written}"
assert (RIDES_SILVER / f"ride_date={first_day}" / "data.parquet").exists(), "файл має лежати в ride_date=<день>/data.parquet"
load_rides_day(rides, second_day, LAKE)

counts = dict(con.sql(f"""
    SELECT ride_date, count(*) FROM read_parquet('{RIDES_SILVER}/*/*.parquet', hive_partitioning = true) GROUP BY ALL
""").fetchall())
assert counts.get(first_day) == expected, f"після двох запусків за {first_day} — {counts.get(first_day)} рядків замість {expected}"
assert counts.get(second_day) == int((rides["ride_date"] == second_day).sum()), "другий день має бути цілим"
file_columns = con.sql(f"SELECT name FROM parquet_schema('{RIDES_SILVER}/ride_date={first_day}/data.parquet')").df()["name"].tolist()
assert "ride_date" not in file_columns, "ride_date має бути лише в імені папки, не у файлі"
print({str(k): v for k, v in sorted(counts.items())})
print("✅ Задача 3 розв'язана")

---
## 9. Архітектура: що де живе

```text
                         «Смачно + Таксі»
 застосунок ──► FastAPI ──► PostgreSQL (OLTP)     ← замовлення зараз: INSERT/UPDATE, транзакції, обмеження
                   │  └───► Redis                 ← кеш, лічильники, черги задач (урок 31)
                   │
                   └─ щоночі вивантаження ──► озеро даних (файли Parquet)
                                               bronze → silver → gold
                                                         │
                                         DuckDB ◄────────┘  ← звіти, аналітика, вітрини для дашборду
                                           │
                                   pandas / графіки / Dash (бонус Pandas)
```

| питання | інструмент |
|---|---|
| «створи замовлення, спиши гроші, не загуби» | PostgreSQL |
| «скільки замовлень за останню хвилину», «кеш звіту на 60 с» | Redis |
| «виторг по районах за рік», «топ кур'єрів» | DuckDB над gold / silver |
| «збережи все як є, розберемось потім» | bronze-шар озера |

### Приклад 9.1. Структура проєкту конвеєра

Пакет, який ми написали за розділ 7:

In [ ]:
for path in sorted(p for p in LAB.rglob("*.py") if "__pycache__" not in p.parts):
    lines = path.read_text(encoding="utf-8").count("\n")
    print(f"{path.relative_to(LAB).as_posix():32} {lines:3} рядків")

- `transform.py` — **чиста** логіка (жодних файлів): її перевіряють тести;
- `pipeline.py` — extract і load: шляхи, файли, ідемпотентний запис;
- `quality.py` — правила якості, однакові для будь-якої таблиці;
- оркестрація (хто й коли запускає `run_pending`) — окремо: cron, Airflow, Dagster, GitHub Actions за розкладом.

### Приклад 9.2. Коли DuckDB не підходить

DuckDB — бібліотека **всередині одного процесу**. Файл `.duckdb` на запис може відкрити лише один процес: другий отримає помилку блокування. Перевіримо: `wh` тримає `warehouse.duckdb`, а окремий процес Python пробує записати в нього.

In [ ]:
WRITER = f"""
import duckdb
duckdb.connect({str(WAREHOUSE)!r}).sql("INSERT INTO dim_date SELECT * FROM dim_date LIMIT 1")
"""
READER = f"""
import duckdb
print(duckdb.connect({str(WAREHOUSE)!r}, read_only=True).sql("SELECT count(*) FROM dim_date").fetchone()[0])
"""


def run_python(code):
    """Запустити код в окремому процесі Python; повернути (код виходу, короткий підсумок)."""
    result = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    if result.returncode == 0:
        return 0, result.stdout.strip()
    last = result.stderr.strip().splitlines()[-1]       # _duckdb.IOException: IO Error: Could not set lock on file "…"
    kind = last.split(":", 1)[0].rsplit(".", 1)[-1]
    message = last.split("Error: ", 1)[-1].split(' "', 1)[0]
    return result.returncode, f"{kind}: {message}"

In [ ]:
print("поки wh відкрите, другий процес пише:  ", run_python(WRITER))
wh.close()
print("wh закрите, другий процес лише читає:  ", run_python(READER))

Поки наш процес тримає файл, інший не може навіть під'єднатися. Коли файл вільний, кілька процесів можуть **читати** його одночасно (`read_only=True`). Звідси правила:

| ✅ DuckDB підходить | ❌ DuckDB не підходить |
|---|---|
| аналітика в ноутбуці, скрипті, конвеєрі | бекенд вебсервісу, куди пишуть сотні користувачів одночасно |
| звіти над Parquet / CSV, data lake | дрібні транзакції «створи замовлення» (приклад 1.4) |
| один процес пише, багато читають готові файли | кілька серверів пишуть в одну базу |
| дані до сотень гігабайтів на одній машині | петабайти на кластері (там — Spark, Trino, BigQuery) |

**Формати таблиць.** Папка з Parquet не має транзакцій: якщо два конвеєри одночасно перезаписують розділ, хтось побачить половину. Цю проблему розв'язують **табличні формати** — шар метаданих поверх Parquet-файлів: **Apache Iceberg**, **Delta Lake** і **DuckLake** (метадані в SQL-базі). Вони дають транзакції, історію версій («як виглядала таблиця вчора»), зміну схеми. **Об'єктне сховище** (Amazon S3, Google Cloud Storage, Cloudflare R2, MinIO) — де такі озера зазвичай лежать: DuckDB читає його тим самим `read_parquet('s3://bucket/lake/silver/orders/*/*.parquet')` через розширення `httpfs`. У ноутбуці ми це не запускаємо — потрібні мережа й обліковий запис, — але шлях і код ті самі, що для папки на диску.

In [ ]:
con.close()
print("готово: усе озеро — у", LAB)

---
## Підсумок

| поняття | що запам'ятати |
|---|---|
| OLTP / OLAP | багато дрібних транзакцій (PostgreSQL) / мало великих сканувань (DuckDB) |
| рядкове / стовпцеве сховище | читати запис цілком / читати лише потрібні стовпці, стискати, обробляти векторами |
| DuckDB | вбудована аналітична база: `duckdb.sql`, `duckdb.connect('файл.duckdb')`, DataFrame за іменем |
| параметри | `?` і список значень; імена таблиць — лише з власного коду |
| `GROUP BY ALL`, `SUMMARIZE`, `PIVOT`, `EXPLAIN ANALYZE` | зручний SQL і план запиту |
| віконні функції, `QUALIFY` | рахувати по групі, не згортаючи рядки; фільтр після вікон |
| Parquet | стовпцевий, типізований, стиснений; projection і predicate pushdown |
| data lake / warehouse / lakehouse | файли будь-які, schema-on-read / таблиці зі схемою / таблиці поверх файлів |
| Hive-розбиття | `ключ=значення/` у шляху; partition pruning; не дрібніше за мегабайти |
| медальйонна архітектура | bronze (як є) → silver (чисто) → gold (вітрини); карантин для відкинутого |
| ідемпотентність | перезапис розділу або DELETE + INSERT у транзакції; повтор не дублює |
| інкрементне завантаження | обробляти лише нові дні: bronze мінус silver |
| зіркова схема | факти (події, числа) + виміри (хто, де, коли) |
| якість і каталог | перевірки як код після кожного завантаження; опис наборів даних |

---
## Самоперевірка

1. Чому той самий `GROUP BY` у DuckDB в десятки разів швидший, ніж у SQLite, а 2000 вставок — навпаки?
2. Чим `duckdb.sql("SELECT … FROM orders")` відрізняється від запиту в PostgreSQL з тим самим текстом? Звідки DuckDB бере `orders`?
3. Чим `QUALIFY` відрізняється від `WHERE` і `HAVING`?
4. Чому Parquet менший за CSV і читається швидше? Що таке projection pushdown і predicate pushdown?
5. Що таке schema-on-read? Чому в bronze читаємо все як текст?
6. Навіщо Hive-розбиття і чим шкодить розбиття «за рестораном і днем»?
7. Що робить завантаження ідемпотентним? Наведи два способи.
8. Чому відкинуті рядки кладуть у карантин, а не просто видаляють?
9. Чим таблиця фактів відрізняється від таблиці вимірів?
10. Коли DuckDB — поганий вибір?

<details>
<summary>Відповіді</summary>

1. Стовпцеве сховище читає лише потрібні стовпці й обробляє їх векторами на всіх ядрах. Зате кожну дрібну вставку треба розкласти по стовпцях — рядкова база тут простіша й швидша.
2. DuckDB працює в процесі Python і знаходить `orders` серед змінних — це DataFrame (replacement scan). PostgreSQL — окремий сервер, він бачить лише свої таблиці.
3. `WHERE` фільтрує рядки до групування, `HAVING` — групи після `GROUP BY`, `QUALIFY` — рядки після віконних функцій.
4. Стовпці одного типу добре стискаються (словник, різниці), типи вже збережені — нічого не треба розбирати. Projection pushdown — читати лише потрібні стовпці; predicate pushdown — фільтрувати під час читання і пропускати групи рядків за їхніми min/max.
5. Схему застосовує той, хто читає, а не той, хто пише. Текстом — щоб жоден рядок не загубився через помилку типу: перетворюємо самі через `TRY_CAST` і бачимо, що саме зламано.
6. Щоб запит із фільтром за ключем читав лише потрібні папки (partition pruning). Надто дрібне розбиття дає тисячі крихітних файлів: метаданих більше, ніж даних, і кожен файл — окреме відкриття.
7. Повторний запуск дає той самий результат. Перезапис розділу дня цілком (через тимчасову папку) або `DELETE` рядків дня + `INSERT` в одній транзакції.
8. Щоб нічого не губити мовчки: можна порахувати, розібратися з джерелом, виправити правило і перезавантажити.
9. Факти — події з числами (довга таблиця: замовлення, сума). Виміри — описи, за якими групують (ресторан, кур'єр, дата).
10. Коли багато клієнтів одночасно пишуть дрібні транзакції (бекенд сервісу) — тут PostgreSQL. Файл `.duckdb` на запис відкриває лише один процес.

</details>

---
## Далі

- Теорія, схеми й порівняльні таблиці: [Бонус. DuckDB і data lake](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m3/bonus_duckdb/).
- Документація DuckDB: [Python API](https://duckdb.org/docs/stable/clients/python/overview), [Parquet](https://duckdb.org/docs/stable/data/parquet/overview), [Hive partitioning](https://duckdb.org/docs/stable/data/partitioning/hive_partitioning), [Friendly SQL](https://duckdb.org/docs/stable/sql/dialect/friendly_sql), [Window functions](https://duckdb.org/docs/stable/sql/functions/window_functions), [Concurrency](https://duckdb.org/docs/stable/connect/concurrency).
- Урок 30 — PostgreSQL: транзакції, параметри й індекси; урок 31 — Redis; бонус Pandas — графіки й Dash, які можна будувати прямо з gold-вітрин.